In [ ]:
# ============================================
# 04 - TRANSFER LEARNING
# Initial Setup
# ============================================

import os
import sys
import json
import time
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import models

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# -----------------------------
# Paths
# -----------------------------

PROJECT_ROOT = "/content/drive/MyDrive/music-genre-capstone"

FEATURE_ROOT = os.path.join(PROJECT_ROOT, "data", "features")

TRAIN_DIR = os.path.join(FEATURE_ROOT, "training_final")
VAL_DIR = os.path.join(FEATURE_ROOT, "validation_final")
TEST_DIR = os.path.join(FEATURE_ROOT, "test_final")

MODEL_DIR = os.path.join(PROJECT_ROOT, "models", "transfer_learning")
RESULT_DIR = os.path.join(PROJECT_ROOT, "results", "transfer_learning")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

# -----------------------------
# Device
# -----------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 50)
print("TRANSFER LEARNING SETUP")
print("=" * 50)

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print()
print("Project root:", PROJECT_ROOT)
print("Training dir:", TRAIN_DIR)
print("Validation dir:", VAL_DIR)
print("Test dir:", TEST_DIR)

print()
print("Train shards:", len([f for f in os.listdir(TRAIN_DIR) if f.endswith(".npz")]))
print("Validation shards:", len([f for f in os.listdir(VAL_DIR) if f.endswith(".npz")]))
print("Test shards:", len([f for f in os.listdir(TEST_DIR) if f.endswith(".npz")]))

TRANSFER LEARNING SETUP
PyTorch version: 2.11.0+cu128
Torchvision version: 0.26.0+cu128
Device: cuda
GPU: Tesla T4

Project root: /content/drive/MyDrive/music-genre-capstone
Training dir: /content/drive/MyDrive/music-genre-capstone/data/features/training_final
Validation dir: /content/drive/MyDrive/music-genre-capstone/data/features/validation_final
Test dir: /content/drive/MyDrive/music-genre-capstone/data/features/test_final



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/music-genre-capstone/data/features/training_final'

In [ ]:
from google.colab import drive
import os
import shutil

# Remove the stale /content/drive directory
if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive")

# Mount Google Drive properly
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Find existing feature directories

FEATURE_ROOT = "/content/drive/MyDrive/music-genre-capstone/data/features"

print("FEATURE ROOT EXISTS:", os.path.exists(FEATURE_ROOT))
print()

if os.path.exists(FEATURE_ROOT):
    print("Folders inside features:")
    for name in sorted(os.listdir(FEATURE_ROOT)):
        path = os.path.join(FEATURE_ROOT, name)
        if os.path.isdir(path):
            print("DIR :", name)
        else:
            print("FILE:", name)
else:
    print("FEATURE ROOT NOT FOUND")

FEATURE ROOT EXISTS: True

Folders inside features:
DIR : test
DIR : test_final
DIR : train
DIR : training
DIR : training_final
DIR : validation
DIR : validation_final


In [ ]:
# ============================================
# Verify final feature cache
# ============================================

TRAIN_DIR = "/content/drive/MyDrive/music-genre-capstone/data/features/training_final"
VAL_DIR = "/content/drive/MyDrive/music-genre-capstone/data/features/validation_final"
TEST_DIR = "/content/drive/MyDrive/music-genre-capstone/data/features/test_final"

train_shards = sorted([
    os.path.join(TRAIN_DIR, f)
    for f in os.listdir(TRAIN_DIR)
    if f.endswith(".npz")
])

val_shards = sorted([
    os.path.join(VAL_DIR, f)
    for f in os.listdir(VAL_DIR)
    if f.endswith(".npz")
])

test_shards = sorted([
    os.path.join(TEST_DIR, f)
    for f in os.listdir(TEST_DIR)
    if f.endswith(".npz")
])

print("Training shards:", len(train_shards))
print("Validation shards:", len(val_shards))
print("Test shards:", len(test_shards))

# Inspect one training shard
data = np.load(train_shards[0])

print()
print("First training shard:", os.path.basename(train_shards[0]))
print("Keys:", list(data.keys()))

for key in data.files:
    print(f"{key}: shape={data[key].shape}, dtype={data[key].dtype}")

Training shards: 40
Validation shards: 6
Test shards: 6

First training shard: shard_000.npz
Keys: ['track_id', 'genre', 'ann_features', 'mel_features']
track_id: shape=(500,), dtype=int64
genre: shape=(500,), dtype=<U13
ann_features: shape=(500, 80), dtype=float32
mel_features: shape=(500, 128, 1292), dtype=float16


In [ ]:
# ============================================
# Lightweight sanity check
# ============================================

print("Training shards:", len(train_shards))
print("Validation shards:", len(val_shards))
print("Test shards:", len(test_shards))

print()
print("Known sample counts:")
print("Training:", 19909)
print("Validation:", 2504)
print("Test:", 2572)

print()
print("Dataset class:", ShardBatchDataset)
print("Status: READY")

Training shards: 40
Validation shards: 6
Test shards: 6

Known sample counts:
Training: 19909
Validation: 2504
Test: 2572

Dataset class: <class '__main__.ShardBatchDataset'>
Status: READY


In [ ]:
# ============================================
# Efficient shard-aware batch loader
# ============================================

class EfficientShardLoader:
    def __init__(self, shard_paths, batch_size=32, shuffle=True):
        self.shard_paths = list(shard_paths)
        self.batch_size = batch_size
        self.shuffle = shuffle

        self.total_samples = 0

        # Read only metadata needed to know shard sizes
        for path in self.shard_paths:
            with np.load(path, mmap_mode=None) as data:
                self.total_samples += data["track_id"].shape[0]

    def __len__(self):
        return (self.total_samples + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        shard_order = list(range(len(self.shard_paths)))

        if self.shuffle:
            random.shuffle(shard_order)

        for shard_idx in shard_order:

            path = self.shard_paths[shard_idx]

            with np.load(path) as data:
                mel = data["mel_features"]
                genres = data["genre"]

                indices = np.arange(len(mel))

                if self.shuffle:
                    np.random.shuffle(indices)

                for start in range(0, len(indices), self.batch_size):

                    batch_indices = indices[start:start + self.batch_size]

                    batch_mel = torch.from_numpy(
                        mel[batch_indices].astype(np.float32)
                    )

                    batch_labels = genres[batch_indices]

                    yield batch_mel, batch_labels


print("EfficientShardLoader created.")
print("Training samples:", 19909)
print("Validation samples:", 2504)
print("Test samples:", 2572)
print("Batch size:", 32)

EfficientShardLoader created.
Training samples: 19909
Validation samples: 2504
Test samples: 2572
Batch size: 32


In [ ]:
# ============================================
# Label encoder + first batch test
# ============================================

# Read only the small genre arrays from each training shard
all_train_labels = []

for path in train_shards:
    with np.load(path) as data:
        all_train_labels.extend(data["genre"].tolist())

label_encoder = LabelEncoder()
label_encoder.fit(all_train_labels)

print("Classes:")
for i, cls in enumerate(label_encoder.classes_):
    print(i, cls)

print()
print("Number of classes:", len(label_encoder.classes_))

# --------------------------------------------
# Test one training batch
# --------------------------------------------

train_loader = EfficientShardLoader(
    train_shards,
    batch_size=32,
    shuffle=True
)

start_time = time.time()

mel_batch, genre_batch = next(iter(train_loader))

batch_time = time.time() - start_time

# Encode labels
label_batch = torch.tensor(
    label_encoder.transform(genre_batch),
    dtype=torch.long
)

print()
print("First batch loaded.")
print("Mel shape:", mel_batch.shape)
print("Mel dtype:", mel_batch.dtype)
print("Label shape:", label_batch.shape)
print("Label dtype:", label_batch.dtype)
print("Batch load time:", round(batch_time, 3), "seconds")
print("Label range:", label_batch.min().item(), "to", label_batch.max().item())

Classes:
0 Blues
1 Classical
2 Country
3 Easy Listening
4 Electronic
5 Experimental
6 Folk
7 Hip-Hop
8 Instrumental
9 International
10 Jazz
11 Old-Time / Historic
12 Pop
13 Rock
14 Soul-RnB
15 Spoken

Number of classes: 16

First batch loaded.
Mel shape: torch.Size([32, 128, 1292])
Mel dtype: torch.float32
Label shape: torch.Size([32])
Label dtype: torch.int64
Batch load time: 0.451 seconds
Label range: 2 to 15


In [ ]:
# ============================================
# ResNet18 Transfer Learning Model
# ============================================

NUM_CLASSES = 16

# Load pretrained ResNet18
weights = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)

# --------------------------------------------
# Adapt first convolution: 3 channels -> 1
# --------------------------------------------

old_conv = resnet.conv1

new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

# Average pretrained RGB filters
with torch.no_grad():
    new_conv.weight[:] = old_conv.weight.mean(
        dim=1,
        keepdim=True
    )

resnet.conv1 = new_conv

# --------------------------------------------
# Replace final classifier
# --------------------------------------------

num_features = resnet.fc.in_features

resnet.fc = nn.Linear(
    num_features,
    NUM_CLASSES
)

resnet = resnet.to(device)

print("Model:", "ResNet18")
print("Input channels:", resnet.conv1.in_channels)
print("Output classes:", NUM_CLASSES)
print("Classifier input features:", num_features)

total_params = sum(p.numel() for p in resnet.parameters())
trainable_params = sum(
    p.numel() for p in resnet.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 50.9MB/s]


Model: ResNet18
Input channels: 1
Output classes: 16
Classifier input features: 512
Total parameters: 11178448
Trainable parameters: 11178448


In [ ]:
# ============================================
# ResNet forward-pass test
# ============================================

resnet.eval()

with torch.no_grad():
    test_input = mel_batch.unsqueeze(1).to(device)

    print("Input shape:", test_input.shape)

    start = time.time()

    test_output = resnet(test_input)

    elapsed = time.time() - start

print("Output shape:", test_output.shape)
print("Forward-pass time:", round(elapsed, 3), "seconds")

assert test_output.shape == (32, 16)

print()
print("Forward pass: SUCCESS")

Input shape: torch.Size([32, 1, 128, 1292])
Output shape: torch.Size([32, 16])
Forward-pass time: 1.811 seconds

Forward pass: SUCCESS


In [ ]:
# ============================================
# Experiment A — Feature Extraction
# Freeze ResNet backbone
# ============================================

for param in resnet.parameters():
    param.requires_grad = False

# Only the new classifier learns
for param in resnet.fc.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in resnet.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in resnet.parameters()
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Frozen parameters:", total_params - trainable_params)

Total parameters: 11178448
Trainable parameters: 8208
Frozen parameters: 11170240


In [ ]:
# ============================================
# Check mel spectrogram value range
# ============================================

path = train_shards[0]

with np.load(path) as data:
    sample_mel = data["mel_features"][0]

print("Shape:", sample_mel.shape)
print("Dtype:", sample_mel.dtype)
print("Min:", sample_mel.min())
print("Max:", sample_mel.max())
print("Mean:", sample_mel.mean())
print("Std:", sample_mel.std())

Shape: (128, 1292)
Dtype: float16
Min: -80.0
Max: 0.0
Mean: -39.62
Std: inf


/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:173: RuntimeWarning: overflow encountered in reduce
  arrmean = umr_sum(arr, axis, dtype, keepdims=True, where=where)


In [ ]:
# ============================================
# Verify mel normalization
# ============================================

mel_float = sample_mel.astype(np.float32)

mel_norm = (mel_float + 80.0) / 80.0
mel_norm = np.clip(mel_norm, 0.0, 1.0)

print("Original:")
print("Min:", mel_float.min())
print("Max:", mel_float.max())
print("Mean:", round(float(mel_float.mean()), 4))
print("Std:", round(float(mel_float.std()), 4))

print()
print("Normalized:")
print("Min:", mel_norm.min())
print("Max:", mel_norm.max())
print("Mean:", round(float(mel_norm.mean()), 4))
print("Std:", round(float(mel_norm.std()), 4))

Original:
Min: -80.0
Max: 0.0
Mean: -39.6348
Std: 10.2844

Normalized:
Min: 0.0
Max: 1.0
Mean: 0.5046
Std: 0.1286


In [ ]:
# ============================================
# Transfer Learning Data Loader
# ============================================

class EfficientShardLoader:
    def __init__(self, shard_paths, label_encoder, batch_size=32, shuffle=True):
        self.shard_paths = list(shard_paths)
        self.label_encoder = label_encoder
        self.batch_size = batch_size
        self.shuffle = shuffle

        self.total_samples = 0

        for path in self.shard_paths:
            with np.load(path) as data:
                self.total_samples += data["track_id"].shape[0]

    def __len__(self):
        return (self.total_samples + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        shard_order = list(range(len(self.shard_paths)))

        if self.shuffle:
            random.shuffle(shard_order)

        for shard_idx in shard_order:
            path = self.shard_paths[shard_idx]

            with np.load(path) as data:
                mel = data["mel_features"]

                if "genre" in data:
                    labels = data["genre"]
                else:
                    labels = data["genres"]

                indices = np.arange(len(mel))

                if self.shuffle:
                    np.random.shuffle(indices)

                for start in range(0, len(indices), self.batch_size):
                    batch_indices = indices[start:start + self.batch_size]

                    # float16 -> float32
                    batch_mel = torch.from_numpy(
                        mel[batch_indices].astype(np.float32)
                    )

                    # [-80, 0] -> [0, 1]
                    batch_mel = (batch_mel + 80.0) / 80.0
                    batch_mel = torch.clamp(batch_mel, 0.0, 1.0)

                    # String labels -> integer labels
                    batch_labels = self.label_encoder.transform(
                        labels[batch_indices]
                    )

                    batch_labels = torch.tensor(
                        batch_labels,
                        dtype=torch.long
                    )

                    yield batch_mel, batch_labels


train_loader = EfficientShardLoader(
    train_shards,
    label_encoder,
    batch_size=32,
    shuffle=True
)

val_loader = EfficientShardLoader(
    val_shards,
    label_encoder,
    batch_size=32,
    shuffle=False
)

test_loader = EfficientShardLoader(
    test_shards,
    label_encoder,
    batch_size=32,
    shuffle=False
)

print("Train samples:", train_loader.total_samples)
print("Validation samples:", val_loader.total_samples)
print("Test samples:", test_loader.total_samples)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train samples: 19909
Validation samples: 2504
Test samples: 2572
Train batches: 623
Validation batches: 79
Test batches: 81


In [ ]:
# ============================================
# Experiment A — Feature Extraction Training
# ============================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    resnet.fc.parameters(),
    lr=0.001
)

num_epochs = 10

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0
best_epoch = 0

best_model_path = os.path.join(
    MODEL_DIR,
    "best_resnet18_feature_extraction.pth"
)

for epoch in range(num_epochs):

    # -------------------------
    # Training
    # -------------------------
    resnet.train()

    # Keep frozen BatchNorm layers from updating
    for module in resnet.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    for batch_mel, batch_labels in train_loader:

        batch_mel = batch_mel.unsqueeze(1).to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        outputs = resnet(batch_mel)
        loss = criterion(outputs, batch_labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_labels.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == batch_labels).sum().item()
        total += batch_labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # -------------------------
    # Validation
    # -------------------------
    resnet.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_mel, batch_labels in val_loader:

            batch_mel = batch_mel.unsqueeze(1).to(device)
            batch_labels = batch_labels.to(device)

            outputs = resnet(batch_mel)
            loss = criterion(outputs, batch_labels)

            val_loss_total += loss.item() * batch_labels.size(0)

            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == batch_labels).sum().item()
            val_total += batch_labels.size(0)

    val_loss = val_loss_total / val_total
    val_acc = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    # -------------------------
    # Save best checkpoint
    # -------------------------
    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch + 1

        torch.save(
            {
                "model_state_dict": resnet.state_dict(),
                "epoch": best_epoch,
                "val_acc": best_val_acc,
                "val_loss": val_loss,
                "num_classes": 16,
                "architecture": "ResNet18 Feature Extraction",
                "input_shape": [1, 128, 1292],
                "normalization": "mel_db [-80,0] -> [0,1]"
            },
            best_model_path
        )

        print("  -> Best model saved")

print()
print("Training complete.")
print("Best epoch:", best_epoch)
print("Best validation accuracy:", round(best_val_acc, 4))
print("Checkpoint:", best_model_path)

Epoch 01/10 | Train Loss: 1.6360 | Train Acc: 0.4966 | Val Loss: 1.6129 | Val Acc: 0.5008 | Time: 154.3s


RuntimeError: Parent directory /content/drive/MyDrive/music-genre-capstone/models/transfer_learning does not exist.

In [ ]:
# ============================================
# Create directories + save current best model
# ============================================

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

torch.save(
    {
        "model_state_dict": resnet.state_dict(),
        "epoch": 1,
        "val_acc": 0.5008,
        "val_loss": 1.6129,
        "num_classes": 16,
        "architecture": "ResNet18 Feature Extraction",
        "input_shape": [1, 128, 1292],
        "normalization": "mel_db [-80,0] -> [0,1]"
    },
    best_model_path
)

print("Checkpoint saved successfully.")
print(best_model_path)

Checkpoint saved successfully.
/content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_feature_extraction.pth


In [ ]:
# ============================================
# Continue Experiment A — Epochs 2 to 10
# ============================================

num_epochs = 10

for epoch in range(2, num_epochs + 1):

    # -------------------------
    # Training
    # -------------------------
    resnet.train()

    # Keep frozen BatchNorm layers from updating
    for module in resnet.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    for batch_mel, batch_labels in train_loader:

        batch_mel = batch_mel.unsqueeze(1).to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        outputs = resnet(batch_mel)
        loss = criterion(outputs, batch_labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_labels.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == batch_labels).sum().item()
        total += batch_labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # -------------------------
    # Validation
    # -------------------------
    resnet.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_mel, batch_labels in val_loader:

            batch_mel = batch_mel.unsqueeze(1).to(device)
            batch_labels = batch_labels.to(device)

            outputs = resnet(batch_mel)
            loss = criterion(outputs, batch_labels)

            val_loss_total += loss.item() * batch_labels.size(0)

            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == batch_labels).sum().item()
            val_total += batch_labels.size(0)

    val_loss = val_loss_total / val_total
    val_acc = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    # -------------------------
    # Save best model
    # -------------------------
    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch

        torch.save(
            {
                "model_state_dict": resnet.state_dict(),
                "epoch": best_epoch,
                "val_acc": best_val_acc,
                "val_loss": val_loss,
                "num_classes": 16,
                "architecture": "ResNet18 Feature Extraction",
                "input_shape": [1, 128, 1292],
                "normalization": "mel_db [-80,0] -> [0,1]"
            },
            best_model_path
        )

        print("  -> Best model saved")

print()
print("Feature extraction training complete.")
print("Best epoch:", best_epoch)
print("Best validation accuracy:", round(best_val_acc, 4))
print("Checkpoint:", best_model_path)

Epoch 02/10 | Train Loss: 1.3900 | Train Acc: 0.5777 | Val Loss: 1.5084 | Val Acc: 0.5375 | Time: 120.7s
  -> Best model saved
Epoch 03/10 | Train Loss: 1.3492 | Train Acc: 0.5847 | Val Loss: 1.4472 | Val Acc: 0.5503 | Time: 121.3s
  -> Best model saved
Epoch 04/10 | Train Loss: 1.3107 | Train Acc: 0.6001 | Val Loss: 1.3857 | Val Acc: 0.5875 | Time: 121.1s
  -> Best model saved
Epoch 05/10 | Train Loss: 1.2843 | Train Acc: 0.6045 | Val Loss: 1.3833 | Val Acc: 0.6090 | Time: 121.8s
  -> Best model saved
Epoch 06/10 | Train Loss: 1.2518 | Train Acc: 0.6143 | Val Loss: 1.3474 | Val Acc: 0.6058 | Time: 123.6s
Epoch 07/10 | Train Loss: 1.2356 | Train Acc: 0.6194 | Val Loss: 1.3970 | Val Acc: 0.5807 | Time: 123.1s
Epoch 08/10 | Train Loss: 1.2293 | Train Acc: 0.6201 | Val Loss: 1.3386 | Val Acc: 0.6010 | Time: 123.2s
Epoch 09/10 | Train Loss: 1.2196 | Train Acc: 0.6245 | Val Loss: 1.3841 | Val Acc: 0.5942 | Time: 124.6s
Epoch 10/10 | Train Loss: 1.2037 | Train Acc: 0.6288 | Val Loss: 1.3823 

In [ ]:
# ============================================
# Experiment A — Test Evaluation
# ============================================

checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=False
)

resnet.load_state_dict(checkpoint["model_state_dict"])
resnet.eval()

test_predictions = []
test_targets = []

test_loss_total = 0.0
test_total = 0

start_time = time.time()

with torch.no_grad():

    for batch_mel, batch_labels in test_loader:

        batch_mel = batch_mel.unsqueeze(1).to(device)
        batch_labels = batch_labels.to(device)

        outputs = resnet(batch_mel)
        loss = criterion(outputs, batch_labels)

        test_loss_total += loss.item() * batch_labels.size(0)
        test_total += batch_labels.size(0)

        predictions = outputs.argmax(dim=1)

        test_predictions.extend(
            predictions.cpu().numpy()
        )

        test_targets.extend(
            batch_labels.cpu().numpy()
        )

test_loss = test_loss_total / test_total
test_accuracy = accuracy_score(
    test_targets,
    test_predictions
)

elapsed = time.time() - start_time

print("Best checkpoint epoch:", checkpoint["epoch"])
print("Best validation accuracy:", checkpoint["val_acc"])
print()
print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy * 100, 2), "%")
print("Test samples:", test_total)
print("Evaluation time:", round(elapsed, 2), "seconds")

Best checkpoint epoch: 10
Best validation accuracy: 0.6134185303514377

Test loss: 1.4986
Test accuracy: 56.8 %
Test samples: 2572
Evaluation time: 27.71 seconds


In [ ]:
# ============================================
# Classification Report + Confusion Matrix
# ============================================

class_names = label_encoder.classes_

report = classification_report(
    test_targets,
    test_predictions,
    target_names=class_names,
    zero_division=0
)

cm = confusion_matrix(
    test_targets,
    test_predictions
)

print(report)

print("Confusion matrix shape:", cm.shape)
print(cm)

                     precision    recall  f1-score   support

              Blues       0.00      0.00      0.00         8
          Classical       0.27      0.85      0.41        62
            Country       0.00      0.00      0.00        18
     Easy Listening       0.00      0.00      0.00         6
         Electronic       0.58      0.87      0.69       632
       Experimental       0.40      0.11      0.17       225
               Folk       0.39      0.20      0.27       152
            Hip-Hop       0.68      0.65      0.67       220
       Instrumental       0.33      0.01      0.01       174
      International       0.27      0.03      0.05       102
               Jazz       0.00      0.00      0.00        39
Old-Time / Historic       0.93      0.80      0.86        51
                Pop       0.00      0.00      0.00       119
               Rock       0.61      0.86      0.71       710
           Soul-RnB       0.00      0.00      0.00        42
             Spoken    

In [ ]:
# ============================================
# Save ResNet18 Feature Extraction Results
# ============================================

report_dict = classification_report(
    test_targets,
    test_predictions,
    target_names=class_names,
    zero_division=0,
    output_dict=True
)

results = {
    "experiment": "ResNet18 Feature Extraction",
    "architecture": "ResNet18",
    "total_parameters": int(total_params),
    "trainable_parameters": int(trainable_params),

    "best_epoch": int(checkpoint["epoch"]),
    "best_val_accuracy": float(checkpoint["val_acc"]),
    "best_val_loss": float(checkpoint["val_loss"]),

    "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss),
    "test_samples": int(test_total),
    "evaluation_time_seconds": float(elapsed),

    "normalization": "mel_db [-80, 0] -> [0, 1]",
    "input_shape": [1, 128, 1292],

    "classification_report": report_dict,
    "confusion_matrix": cm.tolist()
}

results_path = os.path.join(
    RESULT_DIR,
    "resnet18_feature_extraction_results.json"
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print("Results saved:")
print(results_path)

Results saved:
/content/drive/MyDrive/music-genre-capstone/results/transfer_learning/resnet18_feature_extraction_results.json


In [ ]:
# ============================================
# Experiment B — ResNet18 Fine-Tuning
# ============================================

# Start from the best Feature Extraction checkpoint
checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=False
)

resnet.load_state_dict(checkpoint["model_state_dict"])

# Freeze everything first
for param in resnet.parameters():
    param.requires_grad = False

# Unfreeze the final ResNet block
for param in resnet.layer4.parameters():
    param.requires_grad = True

# Keep classifier trainable
for param in resnet.fc.parameters():
    param.requires_grad = True

trainable_params_ft = sum(
    p.numel()
    for p in resnet.parameters()
    if p.requires_grad
)

total_params_ft = sum(
    p.numel()
    for p in resnet.parameters()
)

print("Total parameters:", total_params_ft)
print("Trainable parameters:", trainable_params_ft)
print("Frozen parameters:", total_params_ft - trainable_params_ft)

print()
print("Trainable layers:")

for name, param in resnet.named_parameters():
    if param.requires_grad:
        print(name)

Total parameters: 11178448
Trainable parameters: 8401936
Frozen parameters: 2776512

Trainable layers:
layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.weight
fc.bias


In [ ]:
# ============================================
# Experiment B — Optimizer Configuration
# ============================================

criterion_ft = nn.CrossEntropyLoss()

optimizer_ft = optim.Adam(
    [
        {
            "params": resnet.layer4.parameters(),
            "lr": 1e-4
        },
        {
            "params": resnet.fc.parameters(),
            "lr": 5e-4
        }
    ]
)

num_epochs_ft = 10

best_val_acc_ft = 0.0
best_epoch_ft = 0
best_val_loss_ft = float("inf")

best_model_path_ft = os.path.join(
    MODEL_DIR,
    "best_resnet18_finetuned.pth"
)

print("Optimizer: Adam")
print("Layer4 learning rate: 1e-4")
print("FC learning rate: 5e-4")
print("Epochs:", num_epochs_ft)
print("Checkpoint:", best_model_path_ft)

Optimizer: Adam
Layer4 learning rate: 1e-4
FC learning rate: 5e-4
Epochs: 10
Checkpoint: /content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_finetuned.pth


In [ ]:
# ============================================
# Experiment B — Fine-Tuning
# ============================================

history_ft = []

for epoch in range(1, num_epochs_ft + 1):

    epoch_start = time.time()

    # -------------------------
    # Training
    # -------------------------
    resnet.train()

    # Keep BatchNorm in frozen layers fixed
    for name, module in resnet.named_modules():
        if isinstance(module, nn.BatchNorm2d):
            if not name.startswith("layer4"):
                module.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_mel, batch_labels in train_loader:

        batch_mel = batch_mel.unsqueeze(1).to(device)
        batch_labels = batch_labels.to(device)

        optimizer_ft.zero_grad()

        outputs = resnet(batch_mel)
        loss = criterion_ft(outputs, batch_labels)

        loss.backward()
        optimizer_ft.step()

        running_loss += loss.item() * batch_labels.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == batch_labels).sum().item()
        total += batch_labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # -------------------------
    # Validation
    # -------------------------
    resnet.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_mel, batch_labels in val_loader:

            batch_mel = batch_mel.unsqueeze(1).to(device)
            batch_labels = batch_labels.to(device)

            outputs = resnet(batch_mel)
            loss = criterion_ft(outputs, batch_labels)

            val_loss_total += loss.item() * batch_labels.size(0)

            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == batch_labels).sum().item()
            val_total += batch_labels.size(0)

    val_loss = val_loss_total / val_total
    val_acc = val_correct / val_total

    epoch_time = time.time() - epoch_start

    history_ft.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "time_seconds": epoch_time
    })

    print(
        f"Epoch {epoch:02d}/{num_epochs_ft} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {epoch_time:.1f}s"
    )

    # -------------------------
    # Save best checkpoint
    # -------------------------
    if val_acc > best_val_acc_ft:

        best_val_acc_ft = val_acc
        best_epoch_ft = epoch
        best_val_loss_ft = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": resnet.state_dict(),
            "optimizer_state_dict": optimizer_ft.state_dict(),
            "val_acc": val_acc,
            "val_loss": val_loss,
            "total_params": total_params_ft,
            "trainable_params": trainable_params_ft
        }, best_model_path_ft)

        print("  -> Best model saved")

Epoch 01/10 | Train Loss: 1.6099 | Train Acc: 0.5984 | Val Loss: 1.3309 | Val Acc: 0.5791 | Time: 146.9s
  -> Best model saved
Epoch 02/10 | Train Loss: 1.1489 | Train Acc: 0.6445 | Val Loss: 1.2210 | Val Acc: 0.6386 | Time: 147.9s
  -> Best model saved
Epoch 03/10 | Train Loss: 1.0728 | Train Acc: 0.6691 | Val Loss: 1.1667 | Val Acc: 0.6534 | Time: 145.3s
  -> Best model saved
Epoch 04/10 | Train Loss: 0.9924 | Train Acc: 0.6885 | Val Loss: 1.1271 | Val Acc: 0.6522 | Time: 146.8s
Epoch 05/10 | Train Loss: 0.9243 | Train Acc: 0.7076 | Val Loss: 1.1462 | Val Acc: 0.6661 | Time: 147.9s
  -> Best model saved
Epoch 06/10 | Train Loss: 0.8415 | Train Acc: 0.7314 | Val Loss: 1.3803 | Val Acc: 0.6130 | Time: 150.9s
Epoch 07/10 | Train Loss: 0.7564 | Train Acc: 0.7518 | Val Loss: 1.3142 | Val Acc: 0.6573 | Time: 149.2s
Epoch 08/10 | Train Loss: 0.6623 | Train Acc: 0.7789 | Val Loss: 1.4178 | Val Acc: 0.5942 | Time: 151.2s
Epoch 09/10 | Train Loss: 0.5193 | Train Acc: 0.8270 | Val Loss: 1.5921 

In [ ]:
# ============================================
# Experiment B — Test Evaluation
# ============================================

checkpoint_ft = torch.load(
    best_model_path_ft,
    map_location=device,
    weights_only=False
)

resnet.load_state_dict(checkpoint_ft["model_state_dict"])
resnet.eval()

test_loss_total = 0.0
test_correct = 0
test_total = 0

test_predictions = []
test_targets = []

test_start = time.time()

with torch.no_grad():

    for batch_mel, batch_labels in test_loader:

        batch_mel = batch_mel.unsqueeze(1).to(device)
        batch_labels = batch_labels.to(device)

        outputs = resnet(batch_mel)
        loss = criterion_ft(outputs, batch_labels)

        test_loss_total += loss.item() * batch_labels.size(0)

        predictions = outputs.argmax(dim=1)

        test_correct += (predictions == batch_labels).sum().item()
        test_total += batch_labels.size(0)

        test_predictions.extend(predictions.cpu().numpy())
        test_targets.extend(batch_labels.cpu().numpy())

elapsed_ft = time.time() - test_start

test_loss_ft = test_loss_total / test_total
test_accuracy_ft = test_correct / test_total

print("Best checkpoint epoch:", checkpoint_ft["epoch"])
print("Best validation accuracy:", checkpoint_ft["val_acc"])
print("Test loss:", round(test_loss_ft, 4))
print("Test accuracy:", round(test_accuracy_ft * 100, 2), "%")
print("Test samples:", test_total)
print("Evaluation time:", round(elapsed_ft, 2), "seconds")

Best checkpoint epoch: 5
Best validation accuracy: 0.6661341853035144
Test loss: 1.2868
Test accuracy: 62.83 %
Test samples: 2572
Evaluation time: 20.5 seconds


In [ ]:
# ============================================
# Experiment B — Classification Report
# ============================================

report_ft = classification_report(
    test_targets,
    test_predictions,
    target_names=class_names,
    zero_division=0
)

print(report_ft)

                     precision    recall  f1-score   support

              Blues       0.00      0.00      0.00         8
          Classical       0.64      0.81      0.71        62
            Country       0.00      0.00      0.00        18
     Easy Listening       0.00      0.00      0.00         6
         Electronic       0.64      0.91      0.75       632
       Experimental       0.41      0.40      0.40       225
               Folk       0.32      0.25      0.28       152
            Hip-Hop       0.91      0.63      0.74       220
       Instrumental       0.27      0.02      0.04       174
      International       0.40      0.35      0.38       102
               Jazz       0.48      0.36      0.41        39
Old-Time / Historic       0.93      1.00      0.96        51
                Pop       0.28      0.06      0.10       119
               Rock       0.69      0.86      0.77       710
           Soul-RnB       0.00      0.00      0.00        42
             Spoken    

In [ ]:
# ============================================
# Experiment B — Save Results
# ============================================

report_dict_ft = classification_report(
    test_targets,
    test_predictions,
    target_names=class_names,
    zero_division=0,
    output_dict=True
)

cm_ft = confusion_matrix(
    test_targets,
    test_predictions
)

results_ft = {
    "experiment": "ResNet18 Fine-Tuning",
    "architecture": "ResNet18",
    "total_parameters": int(total_params_ft),
    "trainable_parameters": int(trainable_params_ft),
    "frozen_parameters": int(total_params_ft - trainable_params_ft),
    "best_epoch": int(checkpoint_ft["epoch"]),
    "best_val_accuracy": float(checkpoint_ft["val_acc"]),
    "best_val_loss": float(checkpoint_ft["val_loss"]),
    "test_accuracy": float(test_accuracy_ft),
    "test_loss": float(test_loss_ft),
    "test_samples": int(test_total),
    "evaluation_time_seconds": float(elapsed_ft),
    "normalization": "mel_db [-80, 0] -> [0, 1]",
    "input_shape": [1, 128, 1292],
    "unfrozen_layers": [
        "layer4",
        "fc"
    ],
    "layer4_learning_rate": 1e-4,
    "fc_learning_rate": 5e-4,
    "classification_report": report_dict_ft,
    "confusion_matrix": cm_ft.tolist(),
    "training_history": history_ft
}

results_path_ft = os.path.join(
    RESULT_DIR,
    "resnet18_finetuned_results.json"
)

with open(results_path_ft, "w") as f:
    json.dump(results_ft, f, indent=2)

print("Results saved:")
print(results_path_ft)

Results saved:
/content/drive/MyDrive/music-genre-capstone/results/transfer_learning/resnet18_finetuned_results.json


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import json
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import models, transforms

PROJECT_DIR = "/content/drive/MyDrive/music-genre-capstone"

FEATURE_DIR = os.path.join(PROJECT_DIR, "data", "features")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

print("Project exists:", os.path.exists(PROJECT_DIR))
print("Feature directory exists:", os.path.exists(FEATURE_DIR))
print("Model directory exists:", os.path.exists(MODEL_DIR))

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
Project exists: True
Feature directory exists: True
Model directory exists: True
PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
for folder in ["training_final", "validation_final", "test_final"]:
    path = os.path.join(FEATURE_DIR, folder)

    print(f"\n{folder}")
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        files = sorted([
            f for f in os.listdir(path)
            if f.endswith(".npz")
        ])

        print("NPZ shards:", len(files))
        print("First files:", files[:3])


training_final
Exists: True
NPZ shards: 40
First files: ['shard_000.npz', 'shard_001.npz', 'shard_002.npz']

validation_final
Exists: True
NPZ shards: 6
First files: ['shard_000.npz', 'shard_001.npz', 'shard_002.npz']

test_final
Exists: True
NPZ shards: 6
First files: ['shard_000.npz', 'shard_001.npz', 'shard_002.npz']


In [ ]:
train_dir = os.path.join(FEATURE_DIR, "training_final")

train_files = sorted([
    os.path.join(train_dir, f)
    for f in os.listdir(train_dir)
    if f.endswith(".npz")
])

sample_path = train_files[0]

print("Sample shard:")
print(sample_path)

with np.load(sample_path) as data:
    print("\nKeys:")
    print(data.files)

    for key in data.files:
        print(f"{key}: shape={data[key].shape}, dtype={data[key].dtype}")

    if "mel_features" in data:
        print("\nMel sample shape:", data["mel_features"][0].shape)

Sample shard:
/content/drive/MyDrive/music-genre-capstone/data/features/training_final/shard_000.npz

Keys:
['track_id', 'genre', 'ann_features', 'mel_features']
track_id: shape=(500,), dtype=int64
genre: shape=(500,), dtype=<U13
ann_features: shape=(500, 80), dtype=float32
mel_features: shape=(500, 128, 1292), dtype=float16

Mel sample shape: (128, 1292)


In [ ]:
label_classes = [
    "Blues",
    "Classical",
    "Country",
    "Easy Listening",
    "Electronic",
    "Experimental",
    "Folk",
    "Hip-Hop",
    "Instrumental",
    "International",
    "Jazz",
    "Old-Time / Historic",
    "Pop",
    "Rock",
    "Soul-RnB",
    "Spoken"
]

label_to_idx = {
    label: idx
    for idx, label in enumerate(label_classes)
}

idx_to_label = {
    idx: label
    for label, idx in label_to_idx.items()
}

print("Number of classes:", len(label_classes))
print(label_to_idx)

Number of classes: 16
{'Blues': 0, 'Classical': 1, 'Country': 2, 'Easy Listening': 3, 'Electronic': 4, 'Experimental': 5, 'Folk': 6, 'Hip-Hop': 7, 'Instrumental': 8, 'International': 9, 'Jazz': 10, 'Old-Time / Historic': 11, 'Pop': 12, 'Rock': 13, 'Soul-RnB': 14, 'Spoken': 15}


In [ ]:
def count_samples(shard_dir):
    total = 0

    files = sorted([
        os.path.join(shard_dir, f)
        for f in os.listdir(shard_dir)
        if f.endswith(".npz")
    ])

    for path in files:
        with np.load(path) as data:
            if "mel_features" in data:
                total += len(data["mel_features"])

    return len(files), total


for split in ["training_final", "validation_final", "test_final"]:
    path = os.path.join(FEATURE_DIR, split)
    num_shards, num_samples = count_samples(path)

    print(
        f"{split}: "
        f"{num_shards} shards, "
        f"{num_samples} samples"
    )

KeyboardInterrupt: 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

weights = models.ResNet18_Weights.DEFAULT

print("ResNet18 weights:", weights)
print("Transforms:", weights.transforms())

Device: cuda
ResNet18 weights: ResNet18_Weights.IMAGENET1K_V1
Transforms: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F


class ResNetMelDataset(Dataset):
    def __init__(self, feature_dir, label_to_idx):
        self.feature_dir = feature_dir
        self.label_to_idx = label_to_idx

        self.shard_files = sorted(
            os.path.join(feature_dir, f)
            for f in os.listdir(feature_dir)
            if f.endswith(".npz")
        )

        # Store which shard + sample index each item belongs to
        self.samples = []

        for shard_file in self.shard_files:
            with np.load(shard_file, allow_pickle=True) as data:
                if "genre" in data.files:
                    label_key = "genre"
                elif "genres" in data.files:
                    label_key = "genres"
                else:
                    raise KeyError(f"No genre key found in {shard_file}")

                num_samples = len(data[label_key])

            for i in range(num_samples):
                self.samples.append((shard_file, i))

        # Cache loaded shards
        self._cache = {}

        print(
            f"{os.path.basename(feature_dir)}: "
            f"{len(self.samples)} samples | "
            f"{len(self.shard_files)} shards"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        shard_file, sample_idx = self.samples[idx]

        # Load shard only once
        if shard_file not in self._cache:
            self._cache[shard_file] = np.load(
                shard_file,
                allow_pickle=True
            )

        data = self._cache[shard_file]

        mel = data["mel_features"][sample_idx]

        if "genre" in data.files:
            genre = data["genre"][sample_idx]
        else:
            genre = data["genres"][sample_idx]

        # Exact preprocessing used by ResNet Exp A
        mel = mel.astype(np.float32)
        mel = np.clip(mel, -80.0, 0.0)
        mel = (mel + 80.0) / 80.0

        # [128, 1292] -> [1, 128, 1292]
        mel = torch.from_numpy(mel).unsqueeze(0)

        label = torch.tensor(
            self.label_to_idx[str(genre)],
            dtype=torch.long
        )

        return mel, label

In [ ]:
train_dataset = ResNetMelDataset(
    os.path.join(FEATURE_DIR, "training_final"),
    label_to_idx
)

val_dataset = ResNetMelDataset(
    os.path.join(FEATURE_DIR, "validation_final"),
    label_to_idx
)

test_dataset = ResNetMelDataset(
    os.path.join(FEATURE_DIR, "test_final"),
    label_to_idx
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

KeyboardInterrupt: 

In [ ]:
# Recreate ResNet18
resnet18 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Change first convolution from 3-channel RGB to 1-channel Mel
original_conv1 = resnet18.conv1

resnet18.conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=original_conv1.out_channels,
    kernel_size=original_conv1.kernel_size,
    stride=original_conv1.stride,
    padding=original_conv1.padding,
    bias=False
)

# Replace ImageNet classifier with our 16-class classifier
resnet18.fc = nn.Linear(
    resnet18.fc.in_features,
    len(label_classes)
)

resnet18 = resnet18.to(device)

# Load Exp A checkpoint
resnet18.load_state_dict(
    checkpoint["model_state_dict"]
)

print("ResNet18 Exp A checkpoint loaded successfully.")
print("Input channels:", resnet18.conv1.in_channels)
print("Output classes:", resnet18.fc.out_features)

ResNet18 Exp A checkpoint loaded successfully.
Input channels: 1
Output classes: 16


In [ ]:
print("Checkpoint metadata")
print("===================")
print("Epoch:", checkpoint["epoch"])
print("Val accuracy:", checkpoint["val_acc"])
print("Val loss:", checkpoint["val_loss"])
print("Num classes:", checkpoint["num_classes"])
print("Architecture:", checkpoint["architecture"])
print("Input shape:", checkpoint["input_shape"])
print("Normalization:", checkpoint["normalization"])

Checkpoint metadata
Epoch: 10
Val accuracy: 0.6134185303514377
Val loss: 1.3822709477890414
Num classes: 16
Architecture: ResNet18 Feature Extraction
Input shape: [1, 128, 1292]
Normalization: mel_db [-80,0] -> [0,1]


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class ResNetMelDataset(Dataset):
    def __init__(self, shard_dir, label_to_idx):
        self.shard_files = sorted([
            os.path.join(shard_dir, f)
            for f in os.listdir(shard_dir)
            if f.endswith(".npz")
        ])

        self.label_to_idx = label_to_idx

        # Build index only.
        self.samples = []

        for shard_file in self.shard_files:
            data = np.load(shard_file)

            n = len(data["genre"])

            for i in range(n):
                self.samples.append(
                    (shard_file, i)
                )

            data.close()

        print(
            f"{os.path.basename(shard_dir)}: "
            f"{len(self.samples)} samples"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        shard_file, sample_idx = self.samples[idx]

        data = np.load(shard_file)

        mel = data["mel_features"][sample_idx]
        genre = data["genre"][sample_idx]

        data.close()

        # float32
        mel = mel.astype(np.float32)

        # Exact Exp A normalization:
        # mel dB approximately [-80, 0] -> [0, 1]
        mel = np.clip(mel, -80.0, 0.0)
        mel = (mel + 80.0) / 80.0

        # [128, 1292] -> [1, 128, 1292]
        mel = torch.from_numpy(mel).unsqueeze(0)

        label = self.label_to_idx[genre]

        return mel, torch.tensor(label, dtype=torch.long)

In [ ]:
TRAIN_FEATURE_DIR = "/content/drive/MyDrive/music-genre-capstone/data/features/training_final"
VAL_FEATURE_DIR = "/content/drive/MyDrive/music-genre-capstone/data/features/validation_final"
TEST_FEATURE_DIR = "/content/drive/MyDrive/music-genre-capstone/data/features/test_final"

print("Train:", os.path.exists(TRAIN_FEATURE_DIR))
print("Val:", os.path.exists(VAL_FEATURE_DIR))
print("Test:", os.path.exists(TEST_FEATURE_DIR))

Train: True
Val: True
Test: True


In [ ]:
train_dataset = ResNetMelDataset(
    TRAIN_FEATURE_DIR,
    label_to_idx
)

val_dataset = ResNetMelDataset(
    VAL_FEATURE_DIR,
    label_to_idx
)

test_dataset = ResNetMelDataset(
    TEST_FEATURE_DIR,
    label_to_idx
)

training_final: 19909 samples | label key: 'genre'
validation_final: 2504 samples | label key: 'genres'
test_final: 2572 samples | label key: 'genres'


In [ ]:
# Test one sample
x, y = train_dataset[0]

print("Input shape:", x.shape)
print("Input dtype:", x.dtype)
print("Input min:", x.min().item())
print("Input max:", x.max().item())
print("Label index:", y)
print("Label name:", label_classes[y])

Input shape: torch.Size([1, 128, 1292])
Input dtype: torch.float32
Input min: 0.0
Input max: 1.0
Label index: tensor(7)
Label name: Hip-Hop


In [ ]:
train_dataset = ResNetMelDataset(
    TRAIN_FEATURE_DIR,
    label_to_idx
)

val_dataset = ResNetMelDataset(
    VAL_FEATURE_DIR,
    label_to_idx
)

test_dataset = ResNetMelDataset(
    TEST_FEATURE_DIR,
    label_to_idx
)

training_final: 19909 samples | 40 shards
validation_final: 2504 samples | 6 shards
test_final: 2572 samples | 6 shards


In [ ]:
x, y = train_dataset[0]

print("Input shape:", x.shape)
print("Input dtype:", x.dtype)
print("Input min:", x.min().item())
print("Input max:", x.max().item())
print("Label:", label_classes[y])

Input shape: torch.Size([1, 128, 1292])
Input dtype: torch.float32
Input min: 0.0
Input max: 1.0
Label: Hip-Hop


In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 623
Val batches: 79
Test batches: 81


In [ ]:
# Load existing ResNet18 Exp A checkpoint

RESNET_A_PATH = (
    "/content/drive/MyDrive/music-genre-capstone/"
    "models/transfer_learning/best_resnet18_feature_extraction.pth"
)

checkpoint = torch.load(
    RESNET_A_PATH,
    map_location=device
)

print("Checkpoint loaded.")
print("Epoch:", checkpoint["epoch"])
print("Validation Accuracy:", checkpoint["val_acc"])
print("Validation Loss:", checkpoint["val_loss"])
print("Architecture:", checkpoint["architecture"])
print("Input Shape:", checkpoint["input_shape"])
print("Normalization:", checkpoint["normalization"])

Checkpoint loaded.
Epoch: 10
Validation Accuracy: 0.6134185303514377
Validation Loss: 1.3822709477890414
Architecture: ResNet18 Feature Extraction
Input Shape: [1, 128, 1292]
Normalization: mel_db [-80,0] -> [0,1]


In [ ]:
# Reconstruct ResNet18 exactly as used for Exp A

resnet18 = models.resnet18(
    weights=models.ResNet18_Weights.IMAGENET1K_V1
)

# Exp A uses 1-channel Mel spectrogram input
original_conv1 = resnet18.conv1

resnet18.conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=original_conv1.out_channels,
    kernel_size=original_conv1.kernel_size,
    stride=original_conv1.stride,
    padding=original_conv1.padding,
    bias=False
)

# 16 music genres
resnet18.fc = nn.Linear(
    resnet18.fc.in_features,
    len(label_classes)
)

resnet18 = resnet18.to(device)

# Load trained Exp A weights
resnet18.load_state_dict(
    checkpoint["model_state_dict"]
)

print("ResNet18 loaded successfully.")
print("Input channels:", resnet18.conv1.in_channels)
print("Output classes:", resnet18.fc.out_features)

total_params = sum(p.numel() for p in resnet18.parameters())
trainable_params = sum(
    p.numel() for p in resnet18.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

ResNet18 loaded successfully.
Input channels: 1
Output classes: 16
Total parameters: 11178448
Trainable parameters: 11178448


In [ ]:
# ResNet18 Exp B: fine-tune layer4 + fc only

for param in resnet18.parameters():
    param.requires_grad = False

for param in resnet18.layer4.parameters():
    param.requires_grad = True

for param in resnet18.fc.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel()
    for p in resnet18.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in resnet18.parameters()
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

print("\nTrainable layers:")
for name, param in resnet18.named_parameters():
    if param.requires_grad:
        print(name)

Total parameters: 11178448
Trainable parameters: 8401936

Trainable layers:
layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.weight
fc.bias


In [ ]:
# ============================================
# Experiment B — Optimizer Configuration
# ============================================

criterion_ft = nn.CrossEntropyLoss()

optimizer_ft = optim.Adam(
    [
        {
            "params": resnet18.layer4.parameters(),
            "lr": 1e-4
        },
        {
            "params": resnet18.fc.parameters(),
            "lr": 5e-4
        }
    ]
)

num_epochs_ft = 10

best_val_acc_ft = 0.0
best_epoch_ft = 0
best_val_loss_ft = float("inf")

best_model_path_ft = os.path.join(
    MODEL_DIR,
    "best_resnet18_finetuned.pth"
)

total_params_ft = sum(
    p.numel() for p in resnet18.parameters()
)

trainable_params_ft = sum(
    p.numel()
    for p in resnet18.parameters()
    if p.requires_grad
)

print("Optimizer: Adam")
print("Layer4 learning rate: 1e-4")
print("FC learning rate: 5e-4")
print("Epochs:", num_epochs_ft)
print("Total parameters:", total_params_ft)
print("Trainable parameters:", trainable_params_ft)
print("Checkpoint:", best_model_path_ft)

Optimizer: Adam
Layer4 learning rate: 1e-4
FC learning rate: 5e-4
Epochs: 10
Total parameters: 11178448
Trainable parameters: 8401936
Checkpoint: /content/drive/MyDrive/music-genre-capstone/models/best_resnet18_finetuned.pth


In [ ]:
# ============================================
# Experiment B — Fine-Tuning
# ============================================

history_ft = []

for epoch in range(1, num_epochs_ft + 1):

    epoch_start = time.time()

    # -------------------------
    # Training
    # -------------------------
    resnet18.train()

    # Keep BatchNorm in frozen layers fixed
    for name, module in resnet18.named_modules():
        if isinstance(module, nn.BatchNorm2d):
            if not name.startswith("layer4"):
                module.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_mel, batch_labels in train_loader:

        batch_mel = batch_mel.to(device)
        batch_labels = batch_labels.to(device)

        optimizer_ft.zero_grad()

        outputs = resnet18(batch_mel)
        loss = criterion_ft(outputs, batch_labels)

        loss.backward()
        optimizer_ft.step()

        running_loss += loss.item() * batch_labels.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == batch_labels).sum().item()
        total += batch_labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # -------------------------
    # Validation
    # -------------------------
    resnet18.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_mel, batch_labels in val_loader:

            batch_mel = batch_mel.to(device)
            batch_labels = batch_labels.to(device)

            outputs = resnet18(batch_mel)
            loss = criterion_ft(outputs, batch_labels)

            val_loss_total += loss.item() * batch_labels.size(0)

            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == batch_labels).sum().item()
            val_total += batch_labels.size(0)

    val_loss = val_loss_total / val_total
    val_acc = val_correct / val_total

    epoch_time = time.time() - epoch_start

    history_ft.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "time_seconds": epoch_time
    })

    print(
        f"Epoch {epoch:02d}/{num_epochs_ft} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {epoch_time:.1f}s"
    )

    # -------------------------
    # Save best checkpoint
    # -------------------------
    if val_acc > best_val_acc_ft:

        best_val_acc_ft = val_acc
        best_epoch_ft = epoch
        best_val_loss_ft = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": resnet18.state_dict(),
            "optimizer_state_dict": optimizer_ft.state_dict(),
            "val_acc": val_acc,
            "val_loss": val_loss,
            "total_params": total_params_ft,
            "trainable_params": trainable_params_ft
        }, best_model_path_ft)

        print("  -> Best model saved")

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RESNET_A_PATH = "/content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_feature_extraction.pth"
RESNET_A_EXTENDED_PATH = "/content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_feature_extraction_extended.pth"

# Rebuild exact Exp A architecture
model_a = resnet18(weights=None)

# 1-channel input
model_a.conv1 = nn.Conv2d(
    1, 64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

# 16 classes
model_a.fc = nn.Linear(model_a.fc.in_features, 16)

model_a = model_a.to(device)

# Load existing Exp A checkpoint
checkpoint_a = torch.load(
    RESNET_A_PATH,
    map_location=device,
    weights_only=False
)

model_a.load_state_dict(checkpoint_a["model_state_dict"])

print("Loaded Exp A checkpoint")
print("Previous best epoch:", checkpoint_a["epoch"])
print("Previous best val accuracy:", checkpoint_a["val_acc"])
print("Previous best val loss:", checkpoint_a["val_loss"])

# Freeze entire backbone
for param in model_a.parameters():
    param.requires_grad = False

# Only FC is trainable
for param in model_a.fc.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model_a.parameters()
    if p.requires_grad
)

total_params = sum(p.numel() for p in model_a.parameters())

print("Total params:", total_params)
print("Trainable params:", trainable_params)
print("Trainable layer: fc only")

# Optimizer
optimizer_a = optim.Adam(
    model_a.fc.parameters(),
    lr=1e-4
)

criterion_a = nn.CrossEntropyLoss()

print("Exp A extended training setup ready.")

Loaded Exp A checkpoint
Previous best epoch: 10
Previous best val accuracy: 0.6134185303514377
Previous best val loss: 1.3822709477890414
Total params: 11178448
Trainable params: 8208
Trainable layer: fc only
Exp A extended training setup ready.


In [ ]:
import time
import copy

EXTENDED_EPOCHS_A = 40

best_val_acc_a = checkpoint_a["val_acc"]
best_val_loss_a = checkpoint_a["val_loss"]
best_epoch_a = checkpoint_a["epoch"]

print(f"Starting from historical Epoch {best_epoch_a}")
print(f"Starting best Val Acc: {best_val_acc_a:.4f}")
print(f"Training for {EXTENDED_EPOCHS_A} additional epochs")
print("-" * 70)

for epoch in range(1, EXTENDED_EPOCHS_A + 1):

    start_time = time.time()

    # =========================
    # TRAIN
    # =========================
    model_a.train()

    # Keep frozen BatchNorm layers in eval mode
    for module in model_a.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_mel, batch_labels in train_loader:

        batch_mel = batch_mel.to(device)
        batch_labels = batch_labels.to(device)

        optimizer_a.zero_grad()

        outputs = model_a(batch_mel)
        loss = criterion_a(outputs, batch_labels)

        loss.backward()
        optimizer_a.step()

        running_loss += loss.item() * batch_labels.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == batch_labels).sum().item()
        total += batch_labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # =========================
    # VALIDATION
    # =========================
    model_a.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_mel, batch_labels in val_loader:

            batch_mel = batch_mel.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model_a(batch_mel)
            loss = criterion_a(outputs, batch_labels)

            val_loss_total += loss.item() * batch_labels.size(0)

            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == batch_labels).sum().item()
            val_total += batch_labels.size(0)

    val_loss = val_loss_total / val_total
    val_acc = val_correct / val_total

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch:02d}/{EXTENDED_EPOCHS_A} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {elapsed:.1f}s",
        end=""
    )

    # Save only when validation accuracy improves
    if val_acc > best_val_acc_a:

        best_val_acc_a = val_acc
        best_val_loss_a = val_loss
        best_epoch_a = epoch

        torch.save({
            "epoch": epoch,
            "model_state_dict": model_a.state_dict(),
            "optimizer_state_dict": optimizer_a.state_dict(),
            "val_acc": val_acc,
            "val_loss": val_loss,
            "total_params": total_params,
            "trainable_params": trainable_params,
            "architecture": "ResNet18 Feature Extraction Extended",
            "input_shape": [1, 128, 1292],
            "normalization": "mel_db [-80,0] -> [0,1]",
            "previous_best_epoch": checkpoint_a["epoch"],
            "previous_best_val_acc": checkpoint_a["val_acc"],
            "previous_best_val_loss": checkpoint_a["val_loss"]
        }, RESNET_A_EXTENDED_PATH)

        print(" -> NEW BEST SAVED")
    else:
        print()

print("-" * 70)
print("Extended Exp A training complete.")
print(f"Best Val Accuracy: {best_val_acc_a:.4f}")
print(f"Best Val Loss: {best_val_loss_a:.4f}")
print(f"Best Extended Epoch: {best_epoch_a}")
print(f"Checkpoint: {RESNET_A_EXTENDED_PATH}")

Starting from historical Epoch 10
Starting best Val Acc: 0.6134
Training for 40 additional epochs
----------------------------------------------------------------------


KeyboardInterrupt: 

In [ ]:
import time

print("Testing train loader...")

start = time.time()

batch_mel, batch_labels = next(iter(train_loader))

print("First batch loaded.")
print("Shape:", batch_mel.shape)
print("Labels:", batch_labels.shape)
print("Time:", time.time() - start, "seconds")

Testing train loader...
First batch loaded.
Shape: torch.Size([32, 1, 128, 1292])
Labels: torch.Size([32])
Time: 8.422274827957153 seconds


In [ ]:
import time

model_a.eval()

start = time.time()

for i, (batch_mel, batch_labels) in enumerate(train_loader):
    batch_mel = batch_mel.to(device)
    batch_labels = batch_labels.to(device)

    with torch.no_grad():
        _ = model_a(batch_mel)

    if i == 9:
        break

elapsed = time.time() - start

print(f"10 batches time: {elapsed:.2f}s")
print(f"Average per batch: {elapsed/10:.2f}s")
print(f"Estimated full epoch: {(elapsed/10)*623/60:.2f} minutes")

10 batches time: 48.01s
Average per batch: 4.80s
Estimated full epoch: 49.86 minutes


In [ ]:
import psutil

ram = psutil.virtual_memory()

print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")

Total RAM: 12.67 GB
Available RAM: 9.68 GB


In [ ]:
import numpy as np
import torch
import time
import os

print("Caching training features into RAM...")

start = time.time()

train_mels = []
train_labels = []

for i in range(len(train_dataset)):

    mel, label = train_dataset[i]

    # Store compactly as float16
    train_mels.append(mel.squeeze(0).numpy().astype(np.float16))
    train_labels.append(label)

    if (i + 1) % 2000 == 0:
        print(f"Loaded {i + 1}/{len(train_dataset)}")

train_mels = np.stack(train_mels)
train_labels = np.array(train_labels, dtype=np.int64)

print("\nCache complete.")
print("Mel shape:", train_mels.shape)
print("Mel dtype:", train_mels.dtype)
print("Labels shape:", train_labels.shape)
print("RAM cache size:", train_mels.nbytes / (1024**3), "GB")
print("Time:", round(time.time() - start, 1), "seconds")

Caching training features into RAM...
Loaded 2000/19909


KeyboardInterrupt: 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torchvision.models import resnet18
import time
import psutil

ROOT = "/content/drive/MyDrive/music-genre-capstone"

TRAIN_DIR = f"{ROOT}/data/features/training_final"
VAL_DIR = f"{ROOT}/data/features/validation_final"
TEST_DIR = f"{ROOT}/data/features/test_final"

RESNET_A_PATH = (
    f"{ROOT}/models/transfer_learning/"
    "best_resnet18_feature_extraction.pth"
)

RESNET_A_EXTENDED_PATH = (
    f"{ROOT}/models/transfer_learning/"
    "best_resnet18_feature_extraction_extended.pth"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Train shards:", len(glob.glob(TRAIN_DIR + "/*.npz")))
print("Val shards:", len(glob.glob(VAL_DIR + "/*.npz")))
print("Test shards:", len(glob.glob(TEST_DIR + "/*.npz")))

print("Exp A checkpoint exists:", os.path.exists(RESNET_A_PATH))

ram = psutil.virtual_memory()
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")

Mounted at /content/drive
Device: cuda
GPU: Tesla T4
Train shards: 40
Val shards: 6
Test shards: 6
Exp A checkpoint exists: True
Available RAM: 10.54 GB


In [ ]:
print("Loading training shards into RAM...")
start_time = time.time()

GENRE_TO_ID = {
    "Blues": 0,
    "Classical": 1,
    "Country": 2,
    "Easy Listening": 3,
    "Electronic": 4,
    "Experimental": 5,
    "Folk": 6,
    "Hip-Hop": 7,
    "Instrumental": 8,
    "International": 9,
    "Jazz": 10,
    "Old-Time / Historic": 11,
    "Pop": 12,
    "Rock": 13,
    "Soul-RnB": 14,
    "Spoken": 15
}

train_mel_list = []
train_label_list = []

train_files = sorted(glob.glob(TRAIN_DIR + "/*.npz"))

for i, path in enumerate(train_files):

    data = np.load(path)

    mel = data["mel_features"]
    labels = data["genre"]

    # Convert genre names -> integer class IDs
    label_ids = np.array(
        [GENRE_TO_ID[str(label)] for label in labels],
        dtype=np.int64
    )

    train_mel_list.append(
        mel.astype(np.float16, copy=False)
    )

    train_label_list.append(label_ids)

    print(
        f"Shard {i+1:02d}/{len(train_files)} | "
        f"Samples: {len(labels)}"
    )

train_mels = np.concatenate(
    train_mel_list,
    axis=0
)

train_labels = np.concatenate(
    train_label_list,
    axis=0
)

del train_mel_list
del train_label_list

print("\nTraining cache ready.")
print("Mel shape:", train_mels.shape)
print("Mel dtype:", train_mels.dtype)
print("Labels shape:", train_labels.shape)
print(
    "Cache size:",
    f"{train_mels.nbytes / (1024**3):.2f} GB"
)
print(
    "Loading time:",
    f"{time.time() - start_time:.1f}s"
)

print(
    "Mel range:",
    train_mels.min(),
    "to",
    train_mels.max()
)

print("First 10 labels:", train_labels[:10])

Loading training shards into RAM...
Shard 01/40 | Samples: 500
Shard 02/40 | Samples: 500
Shard 03/40 | Samples: 500
Shard 04/40 | Samples: 500
Shard 05/40 | Samples: 500
Shard 06/40 | Samples: 500
Shard 07/40 | Samples: 500
Shard 08/40 | Samples: 500
Shard 09/40 | Samples: 500
Shard 10/40 | Samples: 500
Shard 11/40 | Samples: 500
Shard 12/40 | Samples: 500
Shard 13/40 | Samples: 500
Shard 14/40 | Samples: 500
Shard 15/40 | Samples: 500
Shard 16/40 | Samples: 500
Shard 17/40 | Samples: 500
Shard 18/40 | Samples: 500
Shard 19/40 | Samples: 500
Shard 20/40 | Samples: 500
Shard 21/40 | Samples: 500
Shard 22/40 | Samples: 500
Shard 23/40 | Samples: 500
Shard 24/40 | Samples: 500
Shard 25/40 | Samples: 500
Shard 26/40 | Samples: 500
Shard 27/40 | Samples: 500
Shard 28/40 | Samples: 500
Shard 29/40 | Samples: 500
Shard 30/40 | Samples: 500
Shard 31/40 | Samples: 500
Shard 32/40 | Samples: 500
Shard 33/40 | Samples: 500
Shard 34/40 | Samples: 500
Shard 35/40 | Samples: 500
Shard 36/40 | Sampl

In [1]:
# CELL 1 — Reconnect Drive + check local disk + inspect one shard

import os
import glob
import shutil
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/music-genre-capstone"

TRAIN_DIR = f"{DRIVE_ROOT}/data/features/training_final"
VAL_DIR   = f"{DRIVE_ROOT}/data/features/validation_final"
TEST_DIR  = f"{DRIVE_ROOT}/data/features/test_final"

MODEL_DIR = f"{DRIVE_ROOT}/models/transfer_learning"

train_files = sorted(glob.glob(f"{TRAIN_DIR}/*.npz"))
val_files   = sorted(glob.glob(f"{VAL_DIR}/*.npz"))
test_files  = sorted(glob.glob(f"{TEST_DIR}/*.npz"))

disk = shutil.disk_usage("/content")

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print()
print("Train shards:", len(train_files))
print("Val shards:", len(val_files))
print("Test shards:", len(test_files))

print()
print("Local /content free:", round(disk.free / (1024**3), 2), "GB")

total_train = sum(os.path.getsize(f) for f in train_files)
total_val   = sum(os.path.getsize(f) for f in val_files)
total_test  = sum(os.path.getsize(f) for f in test_files)

print("Train shard size:", round(total_train / (1024**3), 2), "GB")
print("Val shard size:", round(total_val / (1024**3), 2), "GB")
print("Test shard size:", round(total_test / (1024**3), 2), "GB")

# Inspect one shard only
sample = np.load(train_files[0])

print()
print("First shard keys:", sample.files)
print("Mel shape:", sample["mel_features"].shape)
print("Mel dtype:", sample["mel_features"].dtype)
print("Mel min:", sample["mel_features"].min())
print("Mel max:", sample["mel_features"].max())
print("Genre example:", sample["genre"][0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4

Train shards: 40
Val shards: 6
Test shards: 6

Local /content free: 59.06 GB
Train shard size: 6.14 GB
Val shard size: 0.77 GB
Test shard size: 0.79 GB

First shard keys: ['track_id', 'genre', 'ann_features', 'mel_features']
Mel shape: (500, 128, 1292)
Mel dtype: float16
Mel min: -80.0
Mel max: 0.0
Genre example: Hip-Hop


In [2]:
# CELL 2 — Copy feature shards from Drive to local Colab SSD

import os
import glob
import shutil
import time

LOCAL_ROOT = "/content/music_features"

LOCAL_TRAIN = f"{LOCAL_ROOT}/training_final"
LOCAL_VAL   = f"{LOCAL_ROOT}/validation_final"
LOCAL_TEST  = f"{LOCAL_ROOT}/test_final"

os.makedirs(LOCAL_TRAIN, exist_ok=True)
os.makedirs(LOCAL_VAL, exist_ok=True)
os.makedirs(LOCAL_TEST, exist_ok=True)

def copy_shards(src_dir, dst_dir, name):
    files = sorted(glob.glob(f"{src_dir}/*.npz"))

    print(f"\nCopying {name}: {len(files)} shards")
    start = time.time()

    for i, src in enumerate(files, 1):
        dst = os.path.join(dst_dir, os.path.basename(src))

        if not os.path.exists(dst):
            shutil.copy2(src, dst)

        if i == 1 or i % 5 == 0 or i == len(files):
            print(f"{i}/{len(files)} copied")

    elapsed = time.time() - start
    print(f"{name} done in {elapsed/60:.2f} minutes")

copy_shards(TRAIN_DIR, LOCAL_TRAIN, "TRAIN")
copy_shards(VAL_DIR, LOCAL_VAL, "VALIDATION")
copy_shards(TEST_DIR, LOCAL_TEST, "TEST")

print("\nLocal cache ready.")

local_train_files = sorted(glob.glob(f"{LOCAL_TRAIN}/*.npz"))
local_val_files   = sorted(glob.glob(f"{LOCAL_VAL}/*.npz"))
local_test_files  = sorted(glob.glob(f"{LOCAL_TEST}/*.npz"))

print("Local train shards:", len(local_train_files))
print("Local val shards:", len(local_val_files))
print("Local test shards:", len(local_test_files))

free_gb = shutil.disk_usage("/content").free / (1024**3)
print("Remaining /content free:", round(free_gb, 2), "GB")


Copying TRAIN: 40 shards
1/40 copied
5/40 copied
10/40 copied
15/40 copied
20/40 copied
25/40 copied
30/40 copied
35/40 copied
40/40 copied
TRAIN done in 1.06 minutes

Copying VALIDATION: 6 shards
1/6 copied
5/6 copied
6/6 copied
VALIDATION done in 0.67 minutes

Copying TEST: 6 shards
1/6 copied
5/6 copied
6/6 copied
TEST done in 0.51 minutes

Local cache ready.
Local train shards: 40
Local val shards: 6
Local test shards: 6
Remaining /content free: 49.79 GB


In [3]:
# CELL 3 — Local SSD shard loader + speed test

import numpy as np
import torch
import time

GENRE_TO_ID = {
    "Blues": 0,
    "Classical": 1,
    "Country": 2,
    "Easy Listening": 3,
    "Electronic": 4,
    "Experimental": 5,
    "Folk": 6,
    "Hip-Hop": 7,
    "Instrumental": 8,
    "International": 9,
    "Jazz": 10,
    "Old-Time / Historic": 11,
    "Pop": 12,
    "Rock": 13,
    "Soul-RnB": 14,
    "Spoken": 15
}

BATCH_SIZE = 32

def load_shard(path):
    data = np.load(path)

    mel = data["mel_features"].astype(np.float32, copy=False)
    labels_raw = data["genre"]

    # Convert genre names -> integer labels
    labels = np.array(
        [GENRE_TO_ID[str(x)] for x in labels_raw],
        dtype=np.int64
    )

    # ResNet normalization: [-80, 0] -> [0, 1]
    mel = np.clip(mel, -80.0, 0.0)
    mel = (mel + 80.0) / 80.0

    # [N, 128, 1292] -> [N, 1, 128, 1292]
    mel = mel[:, None, :, :]

    return mel, labels


# Load one local shard
start = time.time()

mel_test, labels_test = load_shard(local_train_files[0])

elapsed = time.time() - start

print("Shard loaded in:", round(elapsed, 3), "seconds")
print("Mel shape:", mel_test.shape)
print("Mel dtype:", mel_test.dtype)
print("Label shape:", labels_test.shape)
print("Mel min:", mel_test.min())
print("Mel max:", mel_test.max())

# Move one batch to GPU
x = torch.from_numpy(mel_test[:BATCH_SIZE]).to("cuda")
y = torch.from_numpy(labels_test[:BATCH_SIZE]).to("cuda")

print("GPU batch shape:", x.shape)
print("GPU labels shape:", y.shape)

# Forward-pass benchmark using existing Exp A model if available
if "model_a" in globals():
    model_a.eval()

    torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        for _ in range(20):
            _ = model_a(x)

    torch.cuda.synchronize()
    elapsed = time.time() - start

    print("20 forward passes:", round(elapsed, 3), "seconds")
    print("Average forward pass:", round(elapsed / 20, 4), "seconds")
else:
    print("model_a not currently loaded — loader benchmark complete.")

Shard loaded in: 7.214 seconds
Mel shape: (500, 1, 128, 1292)
Mel dtype: float32
Label shape: (500,)
Mel min: 0.0
Mel max: 1.0
GPU batch shape: torch.Size([32, 1, 128, 1292])
GPU labels shape: torch.Size([32])
model_a not currently loaded — loader benchmark complete.


In [4]:
# CELL 4 — Convert NPZ shards -> fast local PyTorch shards
# This is done ONCE. Training will use the .pt files afterward.

import os
import glob
import time
import torch
import numpy as np

FAST_ROOT = "/content/music_features_fast"

FAST_TRAIN = f"{FAST_ROOT}/training_final"
FAST_VAL   = f"{FAST_ROOT}/validation_final"
FAST_TEST  = f"{FAST_ROOT}/test_final"

os.makedirs(FAST_TRAIN, exist_ok=True)
os.makedirs(FAST_VAL, exist_ok=True)
os.makedirs(FAST_TEST, exist_ok=True)


def convert_shards(src_files, dst_dir, name, label_key):
    print(f"\nConverting {name}: {len(src_files)} shards")

    total_start = time.time()

    for i, src in enumerate(src_files, 1):
        filename = os.path.splitext(os.path.basename(src))[0] + ".pt"
        dst = os.path.join(dst_dir, filename)

        # Skip if already converted
        if os.path.exists(dst):
            print(f"{i}/{len(src_files)} already exists")
            continue

        data = np.load(src)

        # Load mel
        mel = data["mel_features"].astype(np.float32, copy=False)

        # Normalize exactly as ResNet pipeline
        mel = np.clip(mel, -80.0, 0.0)
        mel = (mel + 80.0) / 80.0

        # Add channel dimension
        # [N,128,1292] -> [N,1,128,1292]
        mel = mel[:, None, :, :]

        # Store normalized mel as float16 to save disk
        mel_tensor = torch.from_numpy(mel).half()

        # Labels
        raw_labels = data[label_key]

        labels = np.array(
            [GENRE_TO_ID[str(x)] for x in raw_labels],
            dtype=np.int64
        )

        label_tensor = torch.from_numpy(labels)

        torch.save(
            {
                "mel": mel_tensor,
                "labels": label_tensor
            },
            dst
        )

        print(f"{i}/{len(src_files)} converted")

    elapsed = time.time() - total_start
    print(f"{name} conversion complete: {elapsed/60:.2f} minutes")


convert_shards(
    local_train_files,
    FAST_TRAIN,
    "TRAIN",
    "genre"
)

convert_shards(
    local_val_files,
    FAST_VAL,
    "VALIDATION",
    "genres"
)

convert_shards(
    local_test_files,
    FAST_TEST,
    "TEST",
    "genres"
)


fast_train_files = sorted(glob.glob(f"{FAST_TRAIN}/*.pt"))
fast_val_files   = sorted(glob.glob(f"{FAST_VAL}/*.pt"))
fast_test_files  = sorted(glob.glob(f"{FAST_TEST}/*.pt"))

print("\nFAST CACHE READY")
print("Train:", len(fast_train_files))
print("Val:", len(fast_val_files))
print("Test:", len(fast_test_files))

import shutil
free_gb = shutil.disk_usage("/content").free / (1024**3)
print("Remaining /content free:", round(free_gb, 2), "GB")


Converting TRAIN: 40 shards
1/40 converted
2/40 converted
3/40 converted
4/40 converted
5/40 converted
6/40 converted
7/40 converted
8/40 converted
9/40 converted
10/40 converted
11/40 converted
12/40 converted
13/40 converted
14/40 converted
15/40 converted
16/40 converted
17/40 converted
18/40 converted
19/40 converted
20/40 converted
21/40 converted
22/40 converted
23/40 converted
24/40 converted
25/40 converted
26/40 converted
27/40 converted
28/40 converted
29/40 converted
30/40 converted
31/40 converted
32/40 converted
33/40 converted
34/40 converted
35/40 converted
36/40 converted
37/40 converted
38/40 converted
39/40 converted
40/40 converted
TRAIN conversion complete: 2.45 minutes

Converting VALIDATION: 6 shards
1/6 converted
2/6 converted
3/6 converted
4/6 converted
5/6 converted
6/6 converted
VALIDATION conversion complete: 0.43 minutes

Converting TEST: 6 shards
1/6 converted
2/6 converted
3/6 converted
4/6 converted
5/6 converted
6/6 converted
TEST conversion complete: 0

In [6]:
# CELL 5B — Fix dtype + benchmark

# Convert batch to float32 before sending to GPU
x = shard["mel"][:32].float().to(DEVICE)
y = shard["labels"][:32].to(DEVICE)

print("Input dtype:", x.dtype)
print("Model dtype:", next(model_a.parameters()).dtype)

# Warmup
model_a.eval()

with torch.no_grad():
    for _ in range(5):
        _ = model_a(x)

torch.cuda.synchronize()

# Benchmark
start = time.time()

with torch.no_grad():
    for _ in range(50):
        _ = model_a(x)

torch.cuda.synchronize()

elapsed = time.time() - start

print()
print("50 forward passes:", round(elapsed, 3), "sec")
print("Average forward:", round(elapsed / 50, 4), "sec")
print("Input shape:", x.shape)
print("Input dtype:", x.dtype)

Input dtype: torch.float32
Model dtype: torch.float32

50 forward passes: 3.911 sec
Average forward: 0.0782 sec
Input shape: torch.Size([32, 1, 128, 1292])
Input dtype: torch.float32


In [7]:
# CELL 6 — Extend Exp A from historical epoch 10 -> 50

import time
import random
import numpy as np
import torch
import torch.nn as nn

BATCH_SIZE = 32
EXTENDED_EPOCHS = 40

BEST_EXTENDED_A = (
    "/content/drive/MyDrive/music-genre-capstone/"
    "models/transfer_learning/best_resnet18_feature_extraction_extended.pth"
)

criterion_a = nn.CrossEntropyLoss()

optimizer_a = torch.optim.Adam(
    model_a.fc.parameters(),
    lr=1e-4
)

best_val_acc = checkpoint_a["val_acc"]
best_val_loss = checkpoint_a["val_loss"]
best_epoch = checkpoint_a["epoch"]

print("Starting Exp A extension")
print("Starting from epoch:", best_epoch)
print("Starting best val accuracy:", best_val_acc)
print("Starting best val loss:", best_val_loss)
print("Additional epochs:", EXTENDED_EPOCHS)
print()

# ---------------------------------------------------------
# Validation function
# ---------------------------------------------------------

def evaluate_fast(model, files):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():

        for path in files:

            shard = torch.load(
                path,
                map_location="cpu",
                weights_only=True
            )

            mel = shard["mel"]
            labels = shard["labels"]

            n = len(labels)

            for start in range(0, n, BATCH_SIZE):

                end = min(start + BATCH_SIZE, n)

                x = mel[start:end].float().to(
                    DEVICE,
                    non_blocking=True
                )

                y = labels[start:end].to(
                    DEVICE,
                    non_blocking=True
                )

                outputs = model(x)

                loss = criterion_a(outputs, y)

                total_loss += loss.item() * len(y)
                total_correct += (
                    outputs.argmax(dim=1) == y
                ).sum().item()

                total_samples += len(y)

    return (
        total_loss / total_samples,
        total_correct / total_samples
    )


# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for extension_epoch in range(1, EXTENDED_EPOCHS + 1):

    actual_epoch = best_epoch + extension_epoch

    epoch_start = time.time()

    model_a.train()

    # Keep frozen backbone frozen
    for param in model_a.parameters():
        param.requires_grad = False

    for param in model_a.fc.parameters():
        param.requires_grad = True

    # Randomize shard order every epoch
    shard_order = list(range(len(fast_train_files)))
    random.shuffle(shard_order)

    total_train_loss = 0.0
    total_train_correct = 0
    total_train_samples = 0

    batch_counter = 0

    for shard_index in shard_order:

        shard = torch.load(
            fast_train_files[shard_index],
            map_location="cpu",
            weights_only=True
        )

        mel = shard["mel"]
        labels = shard["labels"]

        # Shuffle samples inside shard
        indices = torch.randperm(len(labels))

        for start in range(0, len(labels), BATCH_SIZE):

            end = min(start + BATCH_SIZE, len(labels))

            batch_indices = indices[start:end]

            x = mel[batch_indices].float().to(
                DEVICE,
                non_blocking=True
            )

            y = labels[batch_indices].to(
                DEVICE,
                non_blocking=True
            )

            optimizer_a.zero_grad(set_to_none=True)

            outputs = model_a(x)

            loss = criterion_a(outputs, y)

            loss.backward()

            optimizer_a.step()

            total_train_loss += loss.item() * len(y)
            total_train_correct += (
                outputs.argmax(dim=1) == y
            ).sum().item()

            total_train_samples += len(y)

            batch_counter += 1

    train_loss = total_train_loss / total_train_samples
    train_acc = total_train_correct / total_train_samples

    # Validation
    val_loss, val_acc = evaluate_fast(
        model_a,
        fast_val_files
    )

    epoch_time = time.time() - epoch_start

    improved = val_acc > best_val_acc

    if improved:

        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = actual_epoch

        torch.save(
            {
                "model_state_dict": model_a.state_dict(),
                "epoch": actual_epoch,
                "val_acc": val_acc,
                "val_loss": val_loss,
                "num_classes": 16,
                "architecture": "ResNet18 Feature Extraction Extended",
                "input_shape": [1, 128, 1292],
                "normalization": "mel_db [-80,0] -> [0,1]",
                "trainable_params": 8208
            },
            BEST_EXTENDED_A
        )

    status = " -> NEW BEST" if improved else ""

    print(
        f"Epoch {actual_epoch:02d}/50 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {epoch_time:.1f}s"
        f"{status}"
    )

print()
print("Exp A extension complete.")
print("Best epoch:", best_epoch)
print("Best val accuracy:", best_val_acc)
print("Best val loss:", best_val_loss)
print("Checkpoint:", BEST_EXTENDED_A)

Starting Exp A extension
Starting from epoch: 10
Starting best val accuracy: 0.6134185303514377
Starting best val loss: 1.3822709477890414
Additional epochs: 40

Epoch 11/50 | Train Loss: 6.6491 | Train Acc: 0.1957 | Val Loss: 4.8628 | Val Acc: 0.2121 | Time: 102.1s


KeyboardInterrupt: 

In [8]:
# RESTORE HISTORICAL EXP A CHECKPOINT — DO NOT TRAIN YET

import torch
import torch.nn as nn
import torchvision.models as models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_A = (
    "/content/drive/MyDrive/music-genre-capstone/"
    "models/transfer_learning/best_resnet18_feature_extraction.pth"
)

model_a = models.resnet18(weights=None)

model_a.conv1 = nn.Conv2d(
    1, 64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

model_a.fc = nn.Linear(
    model_a.fc.in_features,
    16
)

checkpoint_a = torch.load(
    CHECKPOINT_A,
    map_location=DEVICE,
    weights_only=False
)

model_a.load_state_dict(checkpoint_a["model_state_dict"])
model_a = model_a.to(DEVICE)

# Freeze entire backbone
for param in model_a.parameters():
    param.requires_grad = False

# Only FC trainable
for param in model_a.fc.parameters():
    param.requires_grad = True

# IMPORTANT:
# Backbone stays in EVAL mode so BatchNorm statistics do NOT change.
model_a.eval()
model_a.fc.train()

print("Restored historical Exp A")
print("Epoch:", checkpoint_a["epoch"])
print("Historical Val Accuracy:", checkpoint_a["val_acc"])
print("Historical Val Loss:", checkpoint_a["val_loss"])
print("Trainable params:",
      sum(p.numel() for p in model_a.parameters() if p.requires_grad))

Restored historical Exp A
Epoch: 10
Historical Val Accuracy: 0.6134185303514377
Historical Val Loss: 1.3822709477890414
Trainable params: 8208


In [9]:
# VERIFY HISTORICAL CHECKPOINT ON FAST VALIDATION DATA

criterion_a = nn.CrossEntropyLoss()
BATCH_SIZE = 32

val_loss, val_acc = evaluate_fast(
    model_a,
    fast_val_files
)

print()
print("Checkpoint verification")
print("Val Loss:", val_loss)
print("Val Accuracy:", val_acc)


Checkpoint verification
Val Loss: 1.3822683163058644
Val Accuracy: 0.6134185303514377


In [10]:
# CELL 6 — CORRECTED Exp A EXTENSION: Epoch 11 -> 50

import time
import random
import torch
import torch.nn as nn

BATCH_SIZE = 32
EXTENDED_EPOCHS = 40

BEST_EXTENDED_A = (
    "/content/drive/MyDrive/music-genre-capstone/"
    "models/transfer_learning/best_resnet18_feature_extraction_extended.pth"
)

criterion_a = nn.CrossEntropyLoss()

optimizer_a = torch.optim.Adam(
    model_a.fc.parameters(),
    lr=1e-4
)

# Historical best
best_val_acc = checkpoint_a["val_acc"]
best_val_loss = checkpoint_a["val_loss"]
best_epoch = checkpoint_a["epoch"]

print("Starting CORRECTED Exp A extension")
print("Starting from epoch:", best_epoch)
print("Starting best val accuracy:", best_val_acc)
print("Starting best val loss:", best_val_loss)
print("Additional epochs:", EXTENDED_EPOCHS)
print()


for extension_epoch in range(1, EXTENDED_EPOCHS + 1):

    actual_epoch = 10 + extension_epoch
    epoch_start = time.time()

    # -----------------------------------------------------
    # CRITICAL:
    # Keep entire model in eval mode.
    # This freezes BatchNorm running statistics.
    # -----------------------------------------------------
    model_a.eval()

    # Only FC receives gradients
    model_a.fc.train()

    shard_order = list(range(len(fast_train_files)))
    random.shuffle(shard_order)

    total_train_loss = 0.0
    total_train_correct = 0
    total_train_samples = 0

    for shard_index in shard_order:

        shard = torch.load(
            fast_train_files[shard_index],
            map_location="cpu",
            weights_only=True
        )

        mel = shard["mel"]
        labels = shard["labels"]

        # Shuffle samples inside shard
        indices = torch.randperm(len(labels))

        for start in range(0, len(labels), BATCH_SIZE):

            end = min(start + BATCH_SIZE, len(labels))
            batch_indices = indices[start:end]

            x = mel[batch_indices].float().to(
                DEVICE,
                non_blocking=True
            )

            y = labels[batch_indices].to(
                DEVICE,
                non_blocking=True
            )

            optimizer_a.zero_grad(set_to_none=True)

            outputs = model_a(x)

            loss = criterion_a(outputs, y)

            loss.backward()
            optimizer_a.step()

            total_train_loss += loss.item() * len(y)

            total_train_correct += (
                outputs.argmax(dim=1) == y
            ).sum().item()

            total_train_samples += len(y)

    train_loss = total_train_loss / total_train_samples
    train_acc = total_train_correct / total_train_samples

    # -----------------------------------------------------
    # Validation
    # -----------------------------------------------------

    val_loss, val_acc = evaluate_fast(
        model_a,
        fast_val_files
    )

    epoch_time = time.time() - epoch_start

    improved = val_acc > best_val_acc

    if improved:

        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = actual_epoch

        torch.save(
            {
                "model_state_dict": model_a.state_dict(),
                "epoch": actual_epoch,
                "val_acc": val_acc,
                "val_loss": val_loss,
                "num_classes": 16,
                "architecture": "ResNet18 Feature Extraction Extended",
                "input_shape": [1, 128, 1292],
                "normalization": "mel_db [-80,0] -> [0,1]",
                "trainable_params": 8208
            },
            BEST_EXTENDED_A
        )

    status = " -> NEW BEST" if improved else ""

    print(
        f"Epoch {actual_epoch:02d}/50 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {epoch_time:.1f}s"
        f"{status}"
    )


print()
print("=" * 60)
print("Exp A extension complete")
print("=" * 60)
print("Best epoch:", best_epoch)
print("Best val accuracy:", best_val_acc)
print("Best val loss:", best_val_loss)
print("Checkpoint:", BEST_EXTENDED_A)

Starting CORRECTED Exp A extension
Starting from epoch: 10
Starting best val accuracy: 0.6134185303514377
Starting best val loss: 1.3822709477890414
Additional epochs: 40

Epoch 11/50 | Train Loss: 1.1762 | Train Acc: 0.6382 | Val Loss: 1.2251 | Val Acc: 0.6362 | Time: 90.6s -> NEW BEST
Epoch 12/50 | Train Loss: 1.1597 | Train Acc: 0.6427 | Val Loss: 1.2243 | Val Acc: 0.6354 | Time: 85.6s
Epoch 13/50 | Train Loss: 1.1602 | Train Acc: 0.6448 | Val Loss: 1.2280 | Val Acc: 0.6362 | Time: 86.3s
Epoch 14/50 | Train Loss: 1.1595 | Train Acc: 0.6416 | Val Loss: 1.2245 | Val Acc: 0.6382 | Time: 84.5s -> NEW BEST
Epoch 15/50 | Train Loss: 1.1562 | Train Acc: 0.6411 | Val Loss: 1.2212 | Val Acc: 0.6422 | Time: 85.5s -> NEW BEST
Epoch 16/50 | Train Loss: 1.1595 | Train Acc: 0.6415 | Val Loss: 1.2200 | Val Acc: 0.6422 | Time: 82.3s
Epoch 17/50 | Train Loss: 1.1527 | Train Acc: 0.6473 | Val Loss: 1.2315 | Val Acc: 0.6314 | Time: 84.9s
Epoch 18/50 | Train Loss: 1.1560 | Train Acc: 0.6439 | Val Loss:

In [12]:
# CELL 7B — Locate existing Exp B checkpoint

import os
import glob

ROOT = "/content/drive/MyDrive/music-genre-capstone"

matches = []

for pattern in [
    f"{ROOT}/**/*finetun*.pth",
    f"{ROOT}/**/*resnet18*.pth",
    f"{ROOT}/**/*transfer*.pth",
]:
    matches.extend(glob.glob(pattern, recursive=True))

# Remove duplicates
matches = sorted(set(matches))

print("Possible ResNet/finetuning checkpoints:")

for path in matches:
    print(path)

print()
print("Total found:", len(matches))

Possible ResNet/finetuning checkpoints:
/content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_feature_extraction.pth
/content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_feature_extraction_extended.pth
/content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_finetuned.pth

Total found: 3


In [13]:
# CELL 8 — Verify historical Exp B checkpoint

import torch
import os

EXP_B_PATH = "/content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_finetuned.pth"

checkpoint_b = torch.load(EXP_B_PATH, map_location="cpu", weights_only=False)

print("Checkpoint found:", os.path.exists(EXP_B_PATH))
print("Keys:", checkpoint_b.keys())
print()
print("Epoch:", checkpoint_b.get("epoch"))
print("Val Accuracy:", checkpoint_b.get("val_acc"))
print("Val Loss:", checkpoint_b.get("val_loss"))
print("Total Params:", checkpoint_b.get("total_params"))
print("Trainable Params:", checkpoint_b.get("trainable_params"))

Checkpoint found: True
Keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_acc', 'val_loss', 'total_params', 'trainable_params'])

Epoch: 5
Val Accuracy: 0.6661341853035144
Val Loss: 1.1461982868683225
Total Params: 11178448
Trainable Params: 8401936


In [14]:
# CELL 9 — Rebuild Exp B from historical best checkpoint

import torch
import torch.nn as nn
from torchvision import models

# Paths
EXP_B_PATH = "/content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_finetuned.pth"

# Create ResNet18
model_b = models.resnet18(weights=None)

# 1-channel input
model_b.conv1 = nn.Conv2d(
    1, 64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

# 16-class output
model_b.fc = nn.Linear(model_b.fc.in_features, 16)

# Load historical best checkpoint
checkpoint_b = torch.load(
    EXP_B_PATH,
    map_location=DEVICE,
    weights_only=False
)

model_b.load_state_dict(checkpoint_b["model_state_dict"])
model_b = model_b.to(DEVICE)

# Freeze everything first
for param in model_b.parameters():
    param.requires_grad = False

# Exp B: train layer4 + fc
for param in model_b.layer4.parameters():
    param.requires_grad = True

for param in model_b.fc.parameters():
    param.requires_grad = True

# Count parameters
total_params = sum(p.numel() for p in model_b.parameters())
trainable_params = sum(
    p.numel() for p in model_b.parameters()
    if p.requires_grad
)

print("Exp B model rebuilt successfully")
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Starting from historical epoch:", checkpoint_b["epoch"])
print("Historical best val accuracy:", checkpoint_b["val_acc"])
print("Historical best val loss:", checkpoint_b["val_loss"])

Exp B model rebuilt successfully
Total parameters: 11178448
Trainable parameters: 8401936
Starting from historical epoch: 5
Historical best val accuracy: 0.6661341853035144
Historical best val loss: 1.1461982868683225


In [16]:
# CELL 10 FIX — Use converted .pt validation shards

import glob

FAST_VAL_DIR = "/content/music_features_fast/validation_final"

val_fast_files = sorted(
    glob.glob(f"{FAST_VAL_DIR}/*.pt")
)

print("Fast validation shards:", len(val_fast_files))
print("First shard:", val_fast_files[0])

# Verify checkpoint using the correct .pt files
val_loss_b, val_acc_b = evaluate_fast(
    model_b,
    val_fast_files
)

print()
print("EXP B CHECKPOINT VERIFICATION")
print(f"Validation Loss: {val_loss_b:.6f}")
print(f"Validation Accuracy: {val_acc_b:.4%}")

Fast validation shards: 6
First shard: /content/music_features_fast/validation_final/shard_000.pt

EXP B CHECKPOINT VERIFICATION
Validation Loss: 1.146192
Validation Accuracy: 66.6134%


In [19]:
# CELL 11 — Extend Exp B from historical best Epoch 5 → Epoch 45

import torch
import torch.nn as nn
import time
import os

# --------------------------------------------------
# Optimizer — same historical Exp B learning rates
# --------------------------------------------------

optimizer_b = torch.optim.Adam([
    {
        "params": model_b.layer4.parameters(),
        "lr": 1e-4
    },
    {
        "params": model_b.fc.parameters(),
        "lr": 5e-4
    }
])

# Restore historical optimizer state
optimizer_b.load_state_dict(checkpoint_b["optimizer_state_dict"])

criterion_b = nn.CrossEntropyLoss()

# --------------------------------------------------
# Training configuration
# --------------------------------------------------

START_EPOCH = 6
END_EPOCH = 45

best_val_acc_b = checkpoint_b["val_acc"]
best_val_loss_b = checkpoint_b["val_loss"]
best_epoch_b = checkpoint_b["epoch"]

EXTENDED_B_PATH = (
    "/content/drive/MyDrive/music-genre-capstone/"
    "models/transfer_learning/best_resnet18_finetuned_extended.pth"
)

print("Starting Exp B extension")
print(f"Epochs: {START_EPOCH} → {END_EPOCH}")
print(f"Starting best Val Accuracy: {best_val_acc_b:.4%}")
print(f"Starting best Epoch: {best_epoch_b}")
print()

# --------------------------------------------------
# Training loop
# --------------------------------------------------

for epoch in range(START_EPOCH, END_EPOCH + 1):

    epoch_start = time.time()

    # Entire model stays in eval mode first.
    # This freezes BatchNorm statistics in the frozen layers.
    model_b.eval()

    # Only train layer4 and FC.
    model_b.layer4.train()
    model_b.fc.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for shard_idx, shard_path in enumerate(train_fast_files):

        shard = torch.load(
            shard_path,
            map_location="cpu",
            weights_only=False
        )

        mel = shard["mel"]
        labels = shard["labels"]

        num_samples = len(labels)

        for start in range(0, num_samples, 32):

            end = min(start + 32, num_samples)

            x = mel[start:end].float().to(DEVICE)
            y = labels[start:end].long().to(DEVICE)

            optimizer_b.zero_grad(set_to_none=True)

            outputs = model_b(x)
            loss = criterion_b(outputs, y)

            loss.backward()
            optimizer_b.step()

            running_loss += loss.item() * len(y)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == y).sum().item()
            total += len(y)

        print(
            f"\rEpoch {epoch}/{END_EPOCH} | "
            f"Shard {shard_idx + 1}/{len(train_fast_files)}",
            end=""
        )

    train_loss = running_loss / total
    train_acc = correct / total

    # --------------------------------------------------
    # Validation
    # --------------------------------------------------

    val_loss, val_acc = evaluate_fast(
        model_b,
        val_fast_files
    )

    elapsed = time.time() - epoch_start

    print()
    print(
        f"Epoch {epoch:02d}/{END_EPOCH} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {elapsed:.1f}s",
        end=""
    )

    # --------------------------------------------------
    # Save best checkpoint
    # --------------------------------------------------

    if val_acc > best_val_acc_b:

        best_val_acc_b = val_acc
        best_val_loss_b = val_loss
        best_epoch_b = epoch

        torch.save({
            "epoch": epoch,
            "model_state_dict": model_b.state_dict(),
            "optimizer_state_dict": optimizer_b.state_dict(),
            "val_acc": val_acc,
            "val_loss": val_loss,
            "total_params": sum(
                p.numel() for p in model_b.parameters()
            ),
            "trainable_params": sum(
                p.numel()
                for p in model_b.parameters()
                if p.requires_grad
            ),
        }, EXTENDED_B_PATH)

        print(" -> NEW BEST")
    else:
        print()

print()
print("=" * 60)
print("EXP B EXTENSION COMPLETE")
print("=" * 60)
print(f"Best Epoch: {best_epoch_b}")
print(f"Best Val Accuracy: {best_val_acc_b:.4%}")
print(f"Best Val Loss: {best_val_loss_b:.6f}")
print(f"Checkpoint: {EXTENDED_B_PATH}")

Starting Exp B extension
Epochs: 6 → 45
Starting best Val Accuracy: 66.6134%
Starting best Epoch: 5

Epoch 6/45 | Shard 40/40
Epoch 06/45 | Train Loss: 1.2596 | Train Acc: 0.6132 | Val Loss: 1.0804 | Val Acc: 0.6737 | Time: 92.2s -> NEW BEST
Epoch 7/45 | Shard 40/40
Epoch 07/45 | Train Loss: 1.0097 | Train Acc: 0.6879 | Val Loss: 1.1431 | Val Acc: 0.6613 | Time: 92.7s
Epoch 8/45 | Shard 40/40
Epoch 08/45 | Train Loss: 0.7624 | Train Acc: 0.7671 | Val Loss: 1.3688 | Val Acc: 0.6254 | Time: 89.1s
Epoch 9/45 | Shard 40/40
Epoch 09/45 | Train Loss: 0.5391 | Train Acc: 0.8356 | Val Loss: 1.7891 | Val Acc: 0.5998 | Time: 89.1s
Epoch 10/45 | Shard 40/40
Epoch 10/45 | Train Loss: 0.4246 | Train Acc: 0.8674 | Val Loss: 2.0188 | Val Acc: 0.5839 | Time: 89.1s
Epoch 11/45 | Shard 40/40
Epoch 11/45 | Train Loss: 0.3148 | Train Acc: 0.9019 | Val Loss: 1.9923 | Val Acc: 0.6034 | Time: 88.8s
Epoch 12/45 | Shard 40/40
Epoch 12/45 | Train Loss: 0.2409 | Train Acc: 0.9252 | Val Loss: 2.1268 | Val Acc: 0.

In [18]:
# CELL 11A — Define fast feature shard lists

import glob

FAST_TRAIN_DIR = "/content/music_features_fast/training_final"
FAST_VAL_DIR = "/content/music_features_fast/validation_final"

train_fast_files = sorted(
    glob.glob(f"{FAST_TRAIN_DIR}/*.pt")
)

val_fast_files = sorted(
    glob.glob(f"{FAST_VAL_DIR}/*.pt")
)

print("Training shards:", len(train_fast_files))
print("Validation shards:", len(val_fast_files))

print("First train shard:", train_fast_files[0])
print("First val shard:", val_fast_files[0])

Training shards: 40
Validation shards: 6
First train shard: /content/music_features_fast/training_final/shard_000.pt
First val shard: /content/music_features_fast/validation_final/shard_000.pt


In [20]:
# CELL 12 — Load best Exp A + Exp B checkpoints

import torch
import torch.nn as nn
from torchvision import models

EXP_A_PATH = "/content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_feature_extraction_extended.pth"
EXP_B_PATH = "/content/drive/MyDrive/music-genre-capstone/models/transfer_learning/best_resnet18_finetuned_extended.pth"

def build_resnet18():
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(
        1, 64, kernel_size=7, stride=2, padding=3, bias=False
    )
    model.fc = nn.Linear(model.fc.in_features, 16)
    return model.to(DEVICE)

model_a_best = build_resnet18()
model_b_best = build_resnet18()

ckpt_a = torch.load(EXP_A_PATH, map_location=DEVICE, weights_only=False)
ckpt_b = torch.load(EXP_B_PATH, map_location=DEVICE, weights_only=False)

model_a_best.load_state_dict(ckpt_a["model_state_dict"])
model_b_best.load_state_dict(ckpt_b["model_state_dict"])

print("Exp A best epoch:", ckpt_a["epoch"])
print("Exp A val accuracy:", f"{ckpt_a['val_acc']:.4%}")
print()
print("Exp B best epoch:", ckpt_b["epoch"])
print("Exp B val accuracy:", f"{ckpt_b['val_acc']:.4%}")

Exp A best epoch: 43
Exp A val accuracy: 64.6965%

Exp B best epoch: 6
Exp B val accuracy: 67.3722%


In [21]:
# CELL 13 — Final test evaluation for Exp A and Exp B

import glob
import json
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

FAST_TEST_DIR = "/content/music_features_fast/test_final"

test_fast_files = sorted(
    glob.glob(f"{FAST_TEST_DIR}/*.pt")
)

GENRE_NAMES = [
    "Blues",
    "Classical",
    "Country",
    "Easy Listening",
    "Electronic",
    "Experimental",
    "Folk",
    "Hip-Hop",
    "Instrumental",
    "International",
    "Jazz",
    "Old-Time / Historic",
    "Pop",
    "Rock",
    "Soul-RnB",
    "Spoken"
]

def final_test_evaluate(model, files):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0
    total_samples = 0

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for shard_path in files:

            shard = torch.load(
                shard_path,
                map_location="cpu",
                weights_only=False
            )

            mel = shard["mel"]
            labels = shard["labels"]

            for start in range(0, len(labels), 32):

                end = min(start + 32, len(labels))

                x = mel[start:end].float().to(DEVICE)
                y = labels[start:end].long().to(DEVICE)

                outputs = model(x)
                loss = criterion(outputs, y)

                preds = outputs.argmax(dim=1)

                total_loss += loss.item() * len(y)
                total_samples += len(y)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    accuracy = accuracy_score(all_labels, all_preds)

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_preds,
            average="macro",
            zero_division=0
        )
    )

    precision_weighted, recall_weighted, f1_weighted, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_preds,
            average="weighted",
            zero_division=0
        )
    )

    loss = total_loss / total_samples

    report = classification_report(
        all_labels,
        all_preds,
        target_names=GENRE_NAMES,
        zero_division=0,
        output_dict=True
    )

    return {
        "test_loss": float(loss),
        "test_accuracy": float(accuracy),
        "macro_precision": float(precision_macro),
        "macro_recall": float(recall_macro),
        "macro_f1": float(f1_macro),
        "weighted_precision": float(precision_weighted),
        "weighted_recall": float(recall_weighted),
        "weighted_f1": float(f1_weighted),
        "classification_report": report
    }


print("Evaluating Exp A...")
results_a = final_test_evaluate(model_a_best, test_fast_files)

print("Evaluating Exp B...")
results_b = final_test_evaluate(model_b_best, test_fast_files)

print()
print("=" * 60)
print("EXP A — FINAL TEST")
print("=" * 60)
print(f"Test Loss:       {results_a['test_loss']:.4f}")
print(f"Test Accuracy:   {results_a['test_accuracy']:.4%}")
print(f"Macro F1:        {results_a['macro_f1']:.4f}")
print(f"Weighted F1:     {results_a['weighted_f1']:.4f}")

print()
print("=" * 60)
print("EXP B — FINAL TEST")
print("=" * 60)
print(f"Test Loss:       {results_b['test_loss']:.4f}")
print(f"Test Accuracy:   {results_b['test_accuracy']:.4%}")
print(f"Macro F1:        {results_b['macro_f1']:.4f}")
print(f"Weighted F1:     {results_b['weighted_f1']:.4f}")

# Save results immediately to Drive
RESULT_DIR = "/content/drive/MyDrive/music-genre-capstone/results/transfer_learning"
os.makedirs(RESULT_DIR, exist_ok=True)

with open(
    f"{RESULT_DIR}/resnet_extended_test_results.json",
    "w"
) as f:
    json.dump({
        "exp_a": results_a,
        "exp_b": results_b
    }, f, indent=2)

print()
print("Results saved to:")
print(f"{RESULT_DIR}/resnet_extended_test_results.json")

Evaluating Exp A...
Evaluating Exp B...

EXP A — FINAL TEST
Test Loss:       1.3185
Test Accuracy:   59.8367%
Macro F1:        0.3429
Weighted F1:     0.5613

EXP B — FINAL TEST
Test Loss:       1.2444
Test Accuracy:   63.6858%
Macro F1:        0.3879
Weighted F1:     0.5974

Results saved to:
/content/drive/MyDrive/music-genre-capstone/results/transfer_learning/resnet_extended_test_results.json


In [22]:
# CELL 14 — Save compact ResNet summary

summary = {
    "Exp A": {
        "best_epoch": int(ckpt_a["epoch"]),
        "best_val_accuracy": float(ckpt_a["val_acc"]),
        "best_val_loss": float(ckpt_a["val_loss"]),
        "test_accuracy": results_a["test_accuracy"],
        "test_loss": results_a["test_loss"],
        "macro_f1": results_a["macro_f1"],
        "weighted_f1": results_a["weighted_f1"],
        "trainable_params": 8208
    },
    "Exp B": {
        "best_epoch": int(ckpt_b["epoch"]),
        "best_val_accuracy": float(ckpt_b["val_acc"]),
        "best_val_loss": float(ckpt_b["val_loss"]),
        "test_accuracy": results_b["test_accuracy"],
        "test_loss": results_b["test_loss"],
        "macro_f1": results_b["macro_f1"],
        "weighted_f1": results_b["weighted_f1"],
        "trainable_params": 8401936
    }
}

summary_path = (
    "/content/drive/MyDrive/music-genre-capstone/"
    "results/transfer_learning/resnet_final_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("FINAL RESNET SUMMARY SAVED")
print(summary_path)

FINAL RESNET SUMMARY SAVED
/content/drive/MyDrive/music-genre-capstone/results/transfer_learning/resnet_final_summary.json
